# 🗡️ PixelForge — Prompt-Controlled Pixel Art Character Generator (Conditional DCGAN)

<a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Introduction

**PixelForge** trains a **conditional Deep Convolutional GAN** from scratch on **TinyHero**, a public dataset of
3,648 retro 64×64 RPG character sprites. You describe a character in words, for example
*"a front-facing hero with green hair and a red tunic"*, and the generator draws **brand-new characters** that match.

How the prompt reaches the GAN: TinyHero has no captions, so we **label every sprite automatically** with its
viewing direction (from the folder), hair/headwear colour and outfit colour (from its pixels). A keyword parser
turns the prompt into those labels $y$, and the generator draws $G(z, y)$.

No pretrained image models, diffusion models or external APIs are used: every generated sprite is produced by
the generator network trained in this notebook.

**How to run (Google Colab Free):**
1. `Runtime → Change runtime type → T4 GPU` → Save.
2. `Runtime → Run all`.
3. Full training (100 epochs on all 3,648 images) takes about **25 minutes on an Apple M2** (measured). A T4 should be faster (not measured).
4. The final cell downloads `pixelforge_results.zip` (checkpoints, plots, sprites) for the Streamlit app.

If no GPU is allocated the notebook switches to a short **smoke test** automatically (Section 9) and says so.

## 2. Project objectives

1. Download, validate and preprocess the TinyHero sprite dataset.
2. Derive attribute labels (view, hair colour, outfit colour) automatically and parse text prompts into them.
3. Implement a conditional DCGAN (Generator $G(z, y)$ + Discriminator $D(x, y)$) in PyTorch.
4. Train it adversarially with reproducible seeds, logging, checkpoints and resumable state.
5. Generate new 64×64 RGB characters from text prompts and export them as PNG files.
6. Explore the latent space with interpolation strips and GIFs.
7. Evaluate honestly: losses, progression, real-vs-generated, diversity, memorisation and **prompt controllability**.
8. Export a portable checkpoint that runs on CUDA (Colab), MPS (Apple Silicon) or CPU.

## 3. What is a GAN?

A **Generative Adversarial Network** (Goodfellow et al., 2014) learns to produce samples from a data distribution
$p_{data}$ by pitting two neural networks against each other:

```
             random noise z ~ N(0, I)
                     │
                     ▼
              ┌─────────────┐   fake sprite G(z)
              │  GENERATOR  │ ───────────────────┐
              └─────────────┘                    ▼
                     ▲                    ┌───────────────┐
                     │  "how to fool D"   │ DISCRIMINATOR │ ──► real or fake?
                     └─── gradients ◄──── └───────────────┘
                                                 ▲
                    real sprite x ~ TinyHero ────┘
```

The generator never sees a real image directly. It only learns through the discriminator's gradients:
*"change your pixels in this direction to look more real."*

## 4. Generator versus Discriminator

| | **Generator G** | **Discriminator D** |
|---|---|---|
| Input | latent vector $z \in \mathbb{R}^{100}$, $z\sim\mathcal{N}(0,I)$ | image $x \in \mathbb{R}^{3\times64\times64}$ |
| Output | image $G(z)$ in $[-1,1]$ | one logit (real vs fake score) |
| Goal | make $D(G(z))$ look *real* | separate real from generated images |
| Layers | ConvTranspose2d + BatchNorm + ReLU, Tanh output | Conv2d + BatchNorm + LeakyReLU, raw logit |
| Analogy | counterfeiter | detective |

They improve together: a better detective forces a better counterfeiter.

## 5. Mathematical GAN objective

The original minimax game:

$$
\min_G \max_D \; V(D,G) = \mathbb{E}_{x\sim p_{data}}\big[\log D(x)\big] + \mathbb{E}_{z\sim p_z}\big[\log\big(1 - D(G(z))\big)\big]
$$

For a fixed $G$ the optimal discriminator is $D^*(x) = \dfrac{p_{data}(x)}{p_{data}(x) + p_g(x)}$, and the game is
then equivalent to minimising the Jensen–Shannon divergence between $p_{data}$ and $p_g$; the global optimum is
$p_g = p_{data}$ with $D^* = \tfrac12$.

**In practice** (and in this notebook) we use binary cross-entropy on logits $\ell$, with $\sigma$ the sigmoid:

$$
\mathcal{L}_D = -\big[\,y\log\sigma(\ell_{real}) + (1-y)\cdot 0\,\big] - \log\big(1-\sigma(\ell_{fake})\big), \qquad y = 0.9 \text{ (one-sided label smoothing)}
$$

$$
\mathcal{L}_G = -\log \sigma\big(D(G(z))\big) \quad \text{(non-saturating generator loss)}
$$

The non-saturating loss gives the generator strong gradients early in training, when $D$ easily rejects its samples
(the original $\log(1-D(G(z)))$ would saturate and produce near-zero gradients).

## 6. What is DCGAN?

**DCGAN** (Radford, Metz & Chintala, 2016) is a set of architectural rules that made GANs train reliably on images:

* Replace pooling with **strided convolutions** (D) and **transposed convolutions** (G) so the networks learn
  their own down/up-sampling.
* Use **BatchNorm** in both networks (except the G output layer and the D input layer).
* No fully connected hidden layers.
* **ReLU** in the generator, **Tanh** on its output; **LeakyReLU (0.2)** in the discriminator.
* Adam with **lr = 0.0002, β₁ = 0.5**; weights initialised from $\mathcal{N}(0, 0.02)$.

PixelForge follows these rules exactly at 64×64 resolution:

```
G:  z(100×1×1) → 512×4×4 → 256×8×8 → 128×16×16 → 64×32×32 → 3×64×64 (Tanh)
D:  3×64×64 → 64×32×32 → 128×16×16 → 256×8×8 → 512×4×4 → 1 logit
```

## ⚙️ Setup — configuration (edit this cell only)

All tunable settings live here. Defaults are the full-training configuration for a Colab T4 GPU.

In [ ]:
# ======================= PIXELFORGE CONFIGURATION =======================
EPOCHS            = 100     # 100 = recommended (prompt model); 50 = quicker
BATCH_SIZE        = 64      # 64 on a Colab GPU; use 16-32 on Mac MPS
LEARNING_RATE     = 2e-4    # Adam learning rate for both networks
LATENT_DIM        = 100     # size of the noise vector z
NUM_GENERATED     = 16      # characters to generate at the end (4, 9 or 16 make square grids)
CHECKPOINT_EVERY  = 10      # save a numbered checkpoint every N epochs (latest.pt is saved every epoch)
SAMPLE_EVERY      = 5       # save/display a fixed-noise grid every N epochs
LABEL_SMOOTHING   = 0.1     # one-sided: real label = 0.9 (0 disables)
SEED              = 42
VIEWS             = (0, 1, 2, 3)   # 0=back 1=left 2=front 3=right; (2,) = front-facing only (912 images)

CONDITIONAL       = True    # prompt control: G(z, y) with y = (view, hair colour, outfit colour)
D_COND            = "concat" # how D sees labels: "concat" (works best here) or "projection"
EMA_DECAY         = 0.998   # moving average of generator weights for smoother samples (0 = off)

SMOKE_TEST        = False   # True = 2 quick epochs on 256 images to verify the pipeline
ALLOW_FULL_TRAINING_WITHOUT_CUDA = False   # without a CUDA GPU the notebook falls back to the smoke test
RESUME            = True    # continue from checkpoints/latest.pt if this session already has one
SAVE_TO_DRIVE     = False   # optional: copy the results ZIP to Google Drive (asks for permission)
INCLUDE_FULL_CHECKPOINT_IN_ZIP = True      # latest.pt (~75 MB, G+D+optimizers) for resuming later
# =========================================================================

### Environment: install only missing packages and create the project folder

Colab already ships PyTorch with CUDA, NumPy, Pillow, Matplotlib, pandas and tqdm, so normally
nothing is installed. The project modules are written into `./pixelforge/` with `%%writefile`, so this notebook
needs **no files from your computer**.

In [ ]:
import importlib, importlib.util, os, subprocess, sys
from pathlib import Path

required = {"torch": "torch", "numpy": "numpy", "PIL": "pillow",
            "matplotlib": "matplotlib", "pandas": "pandas", "tqdm": "tqdm"}
missing = [pip for mod, pip in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

IN_COLAB = "google.colab" in sys.modules or importlib.util.find_spec("google.colab") is not None
PROJECT_DIR = Path("pixelforge").resolve()
PROJECT_DIR.mkdir(exist_ok=True)
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("Working directory:", Path.cwd(), "| Colab:", IN_COLAB)


def fresh(name):
    # import (or re-import after %%writefile) a project module
    if name in sys.modules:
        return importlib.reload(sys.modules[name])
    return importlib.import_module(name)

Project modules: configuration/device selection and shared image utilities.

In [ ]:
%%writefile config.py
"""PixelForge configuration, device selection and reproducibility helpers.

Every other module imports its defaults from here, so the CLI scripts, the
Streamlit app and the Colab notebook all agree on hyperparameters and paths.
"""
from __future__ import annotations

import os
import random
from dataclasses import asdict, dataclass, field, fields
from pathlib import Path

import numpy as np
import torch

PROJECT_ROOT = Path(__file__).resolve().parent

# TinyHero (AgaMiko/pixel_character_generator). data.zip is a regular file in the
# repo (not Git LFS), so the raw URL returns the 7.3 MB archive directly.
DATASET_URL = "https://github.com/AgaMiko/pixel_character_generator/raw/master/data.zip"
DATASET_REPO = "https://github.com/AgaMiko/pixel_character_generator"

# Folder names inside data.zip -> viewing direction (verified against the archive).
VIEW_NAMES = {0: "back", 1: "left", 2: "front", 3: "right"}

IMAGE_SIZE = 64
NUM_CHANNELS = 3


@dataclass
class Config:
    # --- data ---
    data_dir: str = str(PROJECT_ROOT / "data")
    views: tuple = (0, 1, 2, 3)            # train on all four directions by default
    background: tuple = (0, 0, 0)         # RGBA sprites are composited onto black (white made this DCGAN mode-collapse, see README)
    subset: int | None = None             # use only N images (smoke tests)
    hflip: bool = False                   # optional horizontal-flip augmentation

    # --- model ---
    conditional: bool = True              # G(z, y) / projection D(x, y) on [view, hair, outfit] labels
    n_classes: tuple = (4, 9, 9)          # classes per attribute (see attributes.py)
    d_cond: str = "concat"                # how D sees labels: "concat" (label maps as input channels) or "projection"
    ema_decay: float = 0.998              # exponential moving average of G weights for sampling (0 = off)
    nz: int = 100                         # latent dimension
    ngf: int = 64                         # generator base channels
    ndf: int = 64                         # discriminator base channels

    # --- optimisation ---
    epochs: int = 50
    batch_size: int = 64
    lr: float = 2e-4
    beta1: float = 0.5
    beta2: float = 0.999
    label_smoothing: float = 0.1          # one-sided: real target = 1 - label_smoothing
    real_label_jitter: float = 0.0        # real targets ~ U(r - j, r + j) with r = 1 - label_smoothing
    fake_label_max: float = 0.0           # fake targets ~ U(0, fake_label_max) (0 = hard zeros)
    instance_noise: float = 0.0           # std of Gaussian noise added to D inputs, annealed to 0 over training
    d_dropout: float = 0.0                # dropout in the discriminator (0 = off)
    d_spectral_norm: bool = False         # spectral normalisation on D's conv weights (replaces D's BatchNorm)
    d_minibatch_std: bool = False         # minibatch std-dev feature in D (anti mode collapse, Karras et al. 2018)
    seed: int = 42
    num_workers: int = 0                  # dataset lives in RAM, workers are unnecessary

    # --- logging / checkpointing ---
    sample_every: int = 5                 # save fixed-noise grid every N epochs
    checkpoint_every: int = 10            # save full checkpoint every N epochs
    keep_last: int = 3                    # keep only the N most recent epoch checkpoints (0 = all)
    num_fixed_samples: int = 64
    checkpoint_dir: str = str(PROJECT_ROOT / "checkpoints")
    output_dir: str = str(PROJECT_ROOT / "outputs")
    device: str = "auto"

    def to_dict(self) -> dict:
        d = asdict(self)
        d["views"] = list(self.views)
        d["background"] = list(self.background)
        d["n_classes"] = list(self.n_classes)
        return d

    @classmethod
    def from_dict(cls, d: dict) -> "Config":
        known = {f.name for f in fields(cls)}
        clean = {k: v for k, v in d.items() if k in known}
        if "views" in clean:
            clean["views"] = tuple(clean["views"])
        if "background" in clean:
            clean["background"] = tuple(clean["background"])
        if "n_classes" in clean:
            clean["n_classes"] = tuple(clean["n_classes"])
        if "conditional" not in d:  # checkpoints from before conditional training existed
            clean["conditional"] = False
        return cls(**clean)


def smoke_test_config(**overrides) -> Config:
    """Small, fast configuration that exercises the whole pipeline."""
    cfg = Config(
        epochs=2,
        batch_size=32,
        subset=256,
        sample_every=1,
        checkpoint_every=1,
        keep_last=1,
        num_fixed_samples=16,
        checkpoint_dir=str(PROJECT_ROOT / "checkpoints" / "smoke"),
        output_dir=str(PROJECT_ROOT / "outputs" / "smoke"),
    )
    for k, v in overrides.items():
        setattr(cfg, k, v)
    return cfg


def get_device(preference: str = "auto") -> torch.device:
    """CUDA (Colab) -> MPS (Apple Silicon) -> CPU, unless a device is forced."""
    if preference and preference != "auto":
        if preference == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("CUDA requested but not available on this machine.")
        if preference == "mps" and not torch.backends.mps.is_available():
            raise RuntimeError("MPS requested but not available on this machine.")
        return torch.device(preference)
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")


def describe_device(device: torch.device) -> str:
    if device.type == "cuda":
        props = torch.cuda.get_device_properties(device)
        return f"CUDA - {props.name} ({props.total_memory / 1024**3:.1f} GB)"
    if device.type == "mps":
        return "MPS - Apple Silicon GPU (Metal Performance Shaders)"
    return "CPU"


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

In [ ]:
%%writefile utils.py
"""Image helpers shared by training, generation, evaluation and the app."""
from __future__ import annotations

from collections import deque
from pathlib import Path

import numpy as np
import torch
from PIL import Image


def denormalize(x: torch.Tensor) -> torch.Tensor:
    """Map generator output from [-1, 1] back to [0, 1]."""
    return ((x.detach().float().cpu() + 1.0) / 2.0).clamp(0.0, 1.0)


def tensor_to_pil(x: torch.Tensor) -> list[Image.Image]:
    """(B,3,H,W) tensor in [-1,1] -> list of RGB PIL images (uint8)."""
    if x.dim() == 3:
        x = x.unsqueeze(0)
    arr = (denormalize(x).permute(0, 2, 3, 1).numpy() * 255.0).round().astype(np.uint8)
    return [Image.fromarray(a) for a in arr]


def upscale(img: Image.Image, scale: int) -> Image.Image:
    """Nearest-neighbour upscaling keeps hard pixel edges (no blur)."""
    if scale == 1:
        return img
    return img.resize((img.width * scale, img.height * scale), Image.NEAREST)


def make_grid(images: list[Image.Image], nrow: int | None = None, padding: int = 2,
              pad_color=(20, 16, 36), scale: int = 1) -> Image.Image:
    """Tile PIL images into a grid. nrow = images per row (default: square)."""
    if not images:
        raise ValueError("make_grid() needs at least one image")
    n = len(images)
    nrow = nrow or int(np.ceil(np.sqrt(n)))
    ncol_rows = int(np.ceil(n / nrow))
    w, h = images[0].size
    mode = images[0].mode
    fill = pad_color if mode == "RGB" else (*pad_color, 0)
    grid = Image.new(mode, (nrow * w + (nrow + 1) * padding, ncol_rows * h + (ncol_rows + 1) * padding), fill)
    for i, im in enumerate(images):
        r, c = divmod(i, nrow)
        grid.paste(im, (padding + c * (w + padding), padding + r * (h + padding)))
    return upscale(grid, scale)


def save_gif(frames: list[Image.Image], path: str | Path, duration_ms: int = 120, bounce: bool = True) -> Path:
    """Save an animated GIF (optionally ping-pong so the loop is seamless)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    seq = frames + frames[-2:0:-1] if bounce and len(frames) > 2 else frames
    seq[0].save(path, save_all=True, append_images=seq[1:], duration=duration_ms, loop=0, disposal=2)
    return path


def remove_background(img: Image.Image, bg=(0, 0, 0), tolerance: int = 24) -> Image.Image:
    """OPTIONAL post-processing: approximate transparency for a generated RGB sprite.

    The GAN only ever outputs RGB. Because training sprites were composited onto a
    solid background colour, generated backgrounds are *approximately* that colour.
    We flood-fill from the image border, marking pixels within `tolerance`
    (max abs channel difference) of `bg` as transparent.

    Limitations: with the default black background, dark outline pixels that touch
    the background can be removed too (lower `tolerance` to keep more); noisy
    backgrounds leave speckles; this is a heuristic, not something the model learned.
    """
    rgb = np.asarray(img.convert("RGB")).astype(np.int16)
    h, w, _ = rgb.shape
    close = (np.abs(rgb - np.array(bg, dtype=np.int16)).max(axis=2) <= tolerance)
    mask = np.zeros((h, w), dtype=bool)
    q = deque((y, x) for y in range(h) for x in (0, w - 1) if close[y, x])
    q.extend((y, x) for x in range(w) for y in (0, h - 1) if close[y, x])
    while q:
        y, x = q.popleft()
        if mask[y, x]:
            continue
        mask[y, x] = True
        for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            ny, nx = y + dy, x + dx
            if 0 <= ny < h and 0 <= nx < w and close[ny, nx] and not mask[ny, nx]:
                q.append((ny, nx))
    alpha = np.where(mask, 0, 255).astype(np.uint8)
    rgba = np.dstack([rgb.astype(np.uint8), alpha])
    return Image.fromarray(rgba)

## 7. Dataset exploration — TinyHero

**Source:** [AgaMiko/pixel_character_generator](https://github.com/AgaMiko/pixel_character_generator) (`data.zip`, 7.3 MB).
Characters were assembled from the **Universal LPC Spritesheet**.
**License:** dual **CC-BY-SA 3.0** and **GNU GPL 3.0** (LPC art rules) — attribution required, share-alike.

What the documentation says: 912 characters × 4 directions = 3,648 images at 64×64.
We **verify** this below instead of assuming it. Verified layout of the archive:

| folder | view |
|---|---|
| `data/0/` | back |
| `data/1/` | left |
| `data/2/` | front |
| `data/3/` | right |

The dataset module below downloads, safely extracts and validates the archive.

In [ ]:
%%writefile dataset.py
"""TinyHero dataset: download, validation, preprocessing and PyTorch Dataset.

Verified structure of data.zip (AgaMiko/pixel_character_generator):
    data/0/*.png  back view   (912 files)
    data/1/*.png  left view   (912 files)
    data/2/*.png  front view  (912 files)
    data/3/*.png  right view  (912 files)
The same file name in each folder is the same character. Every file is a
64x64 RGBA PNG with binary alpha (~78% of pixels fully transparent). The RGB
values under transparent pixels are black. We alpha-composite explicitly onto a
configurable solid colour (Config.background). Black is the default: with a white
background the DCGAN mode-collapsed in every configuration we tried (see README,
"Experiment log"), while black trains stably - matching the original TinyHero notebooks.

Run `python dataset.py` to download, validate, and save statistics + a sample grid.
"""
from __future__ import annotations

import argparse
import json
import shutil
import urllib.request
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset

from config import DATASET_URL, IMAGE_SIZE, VIEW_NAMES, Config

EXPECTED_PER_VIEW = 912


def dataset_root(data_dir: str | Path) -> Path:
    return Path(data_dir) / "tinyhero"


def _safe_extract(zf: zipfile.ZipFile, dest: Path) -> None:
    dest = dest.resolve()
    for member in zf.infolist():
        target = (dest / member.filename).resolve()
        if not str(target).startswith(str(dest)):
            raise RuntimeError(f"Unsafe path in archive: {member.filename}")
    zf.extractall(dest)


def download_tinyhero(data_dir: str | Path, url: str = DATASET_URL, force: bool = False) -> Path:
    """Download and extract TinyHero into <data_dir>/tinyhero/{0,1,2,3}/. Idempotent."""
    data_dir = Path(data_dir)
    root = dataset_root(data_dir)
    if root.exists() and not force and all((root / str(v)).is_dir() for v in VIEW_NAMES):
        return root
    data_dir.mkdir(parents=True, exist_ok=True)
    zip_path = data_dir / "data.zip"
    if not zip_path.exists() or force:
        print(f"Downloading TinyHero from {url} ...")
        tmp = zip_path.with_suffix(".part")
        urllib.request.urlretrieve(url, tmp)
        tmp.rename(zip_path)
    if not zipfile.is_zipfile(zip_path):
        head = zip_path.read_bytes()[:200]
        raise RuntimeError(
            f"{zip_path} is not a valid zip archive (first bytes: {head[:60]!r}). "
            "If it is a Git LFS pointer or an HTML page, delete it and re-download, "
            "or clone the repository with `git clone " + "https://github.com/AgaMiko/pixel_character_generator`."
        )
    staging = data_dir / "_extract"
    if staging.exists():
        shutil.rmtree(staging)
    with zipfile.ZipFile(zip_path) as zf:
        _safe_extract(zf, staging)
    src = staging / "data" if (staging / "data").is_dir() else staging
    if root.exists():
        shutil.rmtree(root)
    shutil.move(str(src), str(root))
    shutil.rmtree(staging, ignore_errors=True)
    print(f"Extracted to {root}")
    return root


def list_images(root: str | Path, views=(0, 1, 2, 3)) -> list[tuple[Path, int]]:
    root = Path(root)
    items = []
    for v in views:
        folder = root / str(v)
        if not folder.is_dir():
            raise FileNotFoundError(f"Missing view folder {folder}. Run `python dataset.py` to download TinyHero.")
        # numeric sort so file 10.png follows 9.png
        files = sorted(folder.glob("*.png"), key=lambda p: (int(p.stem) if p.stem.isdigit() else 10**9, p.stem))
        items.extend((f, v) for f in files)
    return items


def load_rgb(path: str | Path, background=Config.background) -> Image.Image:
    """Open a sprite and alpha-composite it onto a solid background -> RGB."""
    im = Image.open(path)
    if im.mode in ("RGBA", "LA", "P"):
        im = im.convert("RGBA")
        bg = Image.new("RGBA", im.size, (*background, 255))
        im = Image.alpha_composite(bg, im)
    im = im.convert("RGB")
    if im.size != (IMAGE_SIZE, IMAGE_SIZE):
        im = im.resize((IMAGE_SIZE, IMAGE_SIZE), Image.NEAREST)
    return im


def to_tensor(img: Image.Image) -> torch.Tensor:
    """PIL RGB uint8 -> float tensor (3,H,W) normalised to [-1, 1]."""
    arr = torch.from_numpy(np.asarray(img, dtype=np.uint8).copy()).permute(2, 0, 1).float()
    return arr / 127.5 - 1.0


class TinyHeroDataset(Dataset):
    """All sprites are decoded once and kept in RAM as uint8 (3648*64*64*3 = 45 MB)."""

    def __init__(self, root: str | Path, views=(0, 1, 2, 3), background=Config.background,
                 subset: int | None = None, hflip: bool = False, seed: int = 42, with_attributes: bool = False):
        self.root = Path(root)
        items = list_images(self.root, views)
        if not items:
            raise RuntimeError(f"No PNG files found under {self.root}")
        if subset is not None and subset < len(items):
            rng = np.random.default_rng(seed)
            idx = np.sort(rng.choice(len(items), size=subset, replace=False))
            items = [items[i] for i in idx]
        self.paths = [p for p, _ in items]
        self.labels = torch.tensor([v for _, v in items], dtype=torch.long)
        self.hflip = hflip
        self.background = tuple(background)
        self.with_attributes = with_attributes
        if with_attributes:  # [view, hair, outfit] per image, see attributes.py
            from attributes import dataset_labels
            self.attrs = torch.from_numpy(dataset_labels(self.root, items))
        imgs = [np.asarray(load_rgb(p, self.background), dtype=np.uint8) for p in self.paths]
        self.data = torch.from_numpy(np.stack(imgs)).permute(0, 3, 1, 2).contiguous()  # (N,3,64,64) uint8

    def __len__(self) -> int:
        return self.data.shape[0]

    def __getitem__(self, i: int):
        x = self.data[i].float() / 127.5 - 1.0
        flipped = self.hflip and bool(torch.rand(()) < 0.5)
        if flipped:
            x = torch.flip(x, dims=[2])
        if not self.with_attributes:
            return x, self.labels[i]
        y = self.attrs[i].clone()
        if flipped and y[0] in (1, 3):  # mirroring swaps left <-> right
            y[0] = 4 - y[0]
        return x, y

    def as_float01(self) -> torch.Tensor:
        """Whole dataset as float in [0,1] (used for nearest-neighbour search)."""
        return self.data.float() / 255.0


def build_dataset(cfg: Config, download: bool = True) -> TinyHeroDataset:
    root = download_tinyhero(cfg.data_dir) if download else dataset_root(cfg.data_dir)
    return TinyHeroDataset(root, cfg.views, cfg.background, cfg.subset, cfg.hflip, cfg.seed,
                           with_attributes=cfg.conditional)


def make_dataloader(ds: Dataset, batch_size: int, shuffle: bool = True, num_workers: int = 0,
                    seed: int = 42, pin_memory: bool = False, drop_last: bool = True) -> DataLoader:
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers,
                      generator=g, pin_memory=pin_memory, drop_last=drop_last)


def validate_dataset(root: str | Path) -> dict:
    """Inspect every file: count per view, sizes, modes, alpha usage, pairing across views."""
    root = Path(root)
    stats = {"root": str(root), "views": {}, "sizes": Counter(), "modes": Counter(), "formats": Counter()}
    names_per_view = {}
    transparent = []
    for v, name in VIEW_NAMES.items():
        files = list((root / str(v)).glob("*"))
        pngs = [f for f in files if f.suffix.lower() == ".png"]
        names_per_view[v] = {f.stem for f in pngs}
        stats["views"][str(v)] = {"name": name, "count": len(pngs), "non_png": len(files) - len(pngs)}
        for f in pngs:
            with Image.open(f) as im:
                stats["sizes"][f"{im.width}x{im.height}"] += 1
                stats["modes"][im.mode] += 1
                stats["formats"][im.format] += 1
                if im.mode == "RGBA" and len(transparent) < 400:
                    transparent.append(float((np.asarray(im)[..., 3] == 0).mean()))
    common = set.intersection(*names_per_view.values()) if names_per_view else set()
    stats["total_images"] = sum(s["count"] for s in stats["views"].values())
    stats["characters_in_all_views"] = len(common)
    stats["mean_transparent_fraction"] = round(float(np.mean(transparent)), 4) if transparent else None
    stats["sizes"] = dict(stats["sizes"])
    stats["modes"] = dict(stats["modes"])
    stats["formats"] = dict(stats["formats"])
    stats["matches_documentation"] = (
        stats["total_images"] == 4 * EXPECTED_PER_VIEW and stats["sizes"] == {"64x64": 4 * EXPECTED_PER_VIEW}
    )
    return stats


def pixel_statistics(ds: TinyHeroDataset) -> dict:
    x = ds.data.float() / 255.0
    return {
        "num_images": len(ds),
        "tensor_shape": list(ds.data.shape),
        "channel_mean_0_1": [round(float(m), 4) for m in x.mean(dim=(0, 2, 3))],
        "channel_std_0_1": [round(float(s), 4) for s in x.std(dim=(0, 2, 3))],
        "normalized_min": round(float(ds[0][0].min()), 4),
        "normalized_max": round(float(ds[0][0].max()), 4),
        "per_view_counts": {VIEW_NAMES[int(k)]: int(v) for k, v in zip(*torch.unique(ds.labels, return_counts=True))},
        "background_rgb": list(ds.background),
    }


def save_sample_grid(ds: TinyHeroDataset, path: str | Path, n: int = 25, seed: int = 0, scale: int = 3) -> Path:
    from utils import make_grid
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(ds), size=min(n, len(ds)), replace=False)
    imgs = [Image.fromarray(ds.data[i].permute(1, 2, 0).numpy()) for i in idx]
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    make_grid(imgs, nrow=int(np.ceil(np.sqrt(len(imgs)))), scale=scale).save(path)
    return path


def save_view_grid(ds_root: str | Path, path: str | Path, n_chars: int = 6, background=Config.background, scale: int = 3) -> Path:
    """Rows = characters, columns = back/left/front/right."""
    from utils import make_grid
    ds_root = Path(ds_root)
    names = sorted((p.stem for p in (ds_root / "2").glob("*.png")), key=lambda s: int(s) if s.isdigit() else 0)
    rng = np.random.default_rng(1)
    chosen = rng.choice(names, size=min(n_chars, len(names)), replace=False)
    imgs = [load_rgb(ds_root / str(v) / f"{c}.png", background) for c in chosen for v in VIEW_NAMES]
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    make_grid(imgs, nrow=4, scale=scale).save(path)
    return path


def main():
    p = argparse.ArgumentParser(description="Download, validate and visualise TinyHero")
    p.add_argument("--data-dir", default=Config.data_dir)
    p.add_argument("--output-dir", default=Config.output_dir)
    p.add_argument("--force-download", action="store_true")
    args = p.parse_args()

    root = download_tinyhero(args.data_dir, force=args.force_download)
    stats = validate_dataset(root)
    ds = TinyHeroDataset(root)
    stats["pixels"] = pixel_statistics(ds)
    out = Path(args.output_dir)
    (out / "plots").mkdir(parents=True, exist_ok=True)
    (out / "dataset_stats.json").write_text(json.dumps(stats, indent=2))
    save_sample_grid(ds, out / "plots" / "dataset_samples.png", n=25)
    save_view_grid(root, out / "plots" / "dataset_views.png")
    print(json.dumps(stats, indent=2))
    print(f"Saved {out / 'dataset_stats.json'}, {out / 'plots' / 'dataset_samples.png'}, {out / 'plots' / 'dataset_views.png'}")


if __name__ == "__main__":
    main()

In [ ]:
import json
import numpy as np
from IPython.display import Image as IPImage, display

config = fresh("config"); utils = fresh("utils"); dataset = fresh("dataset")
root = dataset.download_tinyhero(config.PROJECT_ROOT / "data")
stats = dataset.validate_dataset(root)
print(json.dumps(stats, indent=2))
assert stats["total_images"] > 0, "Dataset extraction failed"
if not stats["matches_documentation"]:
    print("WARNING: dataset differs from the documented 3,648 64x64 images - see stats above.")

In [ ]:
full_ds = dataset.TinyHeroDataset(root)
pix = dataset.pixel_statistics(full_ds)
stats["pixels"] = pix
Path("outputs/plots").mkdir(parents=True, exist_ok=True)
Path("outputs/dataset_stats.json").write_text(json.dumps(stats, indent=2))
print(json.dumps(pix, indent=2))

dataset.save_sample_grid(full_ds, "outputs/plots/dataset_samples.png", n=25, scale=3)
dataset.save_view_grid(root, "outputs/plots/dataset_views.png")
print("25 random REAL training images:")
display(IPImage("outputs/plots/dataset_samples.png"))
print("Same character from 4 directions (columns: back, left, front, right):")
display(IPImage("outputs/plots/dataset_views.png"))

## 8. Preprocessing

1. **RGBA → RGB.** Every sprite is RGBA with binary alpha (~76 % transparent). We alpha-composite onto an
   explicit solid background, **black** by default. White keeps outlines more visible, but in our experiments
   a white background made this DCGAN **mode-collapse** (all latent vectors → one sprite by epoch 5), while black
   trained stably. This matches the original TinyHero notebooks. Change it with `Config.background`.
2. **Normalise to [-1, 1]:** $x' = x/127.5 - 1$, matching the generator's Tanh output range.
3. **In-memory dataset:** all 3,648 sprites are decoded once into a uint8 tensor (≈45 MB), so the DataLoader needs no workers.
4. **All four views** are used for training to maximise data (front-only would leave just 912 images).

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

sample_path = sorted((root / "2").glob("*.png"))[5]
raw = Image.open(sample_path)
fig, ax = plt.subplots(1, 3, figsize=(9, 3.2))
for a, (im, t) in zip(ax, [(raw, f"original {raw.mode}"),
                           (dataset.load_rgb(sample_path, (255, 255, 255)), "on white (collapsed GAN)"),
                           (dataset.load_rgb(sample_path, cfg_bg := tuple(config.Config().background)),
                            f"on {cfg_bg} (used)")]):
    a.imshow(im, interpolation="nearest"); a.set_title(t); a.axis("off")
plt.show()

loader = dataset.make_dataloader(full_ds, batch_size=BATCH_SIZE)
xb, yb = next(iter(loader))
print("batch:", tuple(xb.shape), xb.dtype, "| range [%.2f, %.2f]" % (xb.min(), xb.max()),
      "| views in batch:", np.bincount(yb.numpy(), minlength=4))

### 8b. Automatic attribute labels (what makes prompts possible)

TinyHero has **no text descriptions**, so we derive labels from the pixels:

| attribute | source | classes |
|---|---|---|
| view | folder name (exact) | back, left, front, right |
| hair / headwear colour | majority colour of rows 9–17 (top of head) | black, brown, blonde, red, white, grey, green, blue, purple |
| outfit colour | majority colour of rows 33–43 (torso) | black, brown, yellow, red, white, grey, green, blue, purple |

Each opaque pixel is classified by HSV rules (dark outline pixels ignored) and the majority wins.
**Limitations:** labels are approximate. A grey helmet counts as grey "hair", and a shirtless character's
"outfit" is their skin colour. The keyword parser maps a prompt to these labels. Words it doesn't know
(e.g. "wizard", "sword") are ignored.

In [ ]:
%%writefile attributes.py
"""Automatic attribute labels for TinyHero + a keyword prompt parser.

TinyHero ships no captions, only the viewing direction (folder name). To make
generation controllable we derive two more labels from the pixels themselves:

* hair  - colour of the top of the head (rows 9-17), i.e. hair or headwear
* outfit - colour of the torso (rows 33-43)

Each opaque pixel in the region is classified into a named colour (HSV rules);
dark outline pixels are ignored; the majority colour wins. Labels are therefore
approximate: a grey helmet is labelled "grey" hair, a two-colour shirt gets its
dominant colour. The prompt parser maps words in a description to these labels.
"""
from __future__ import annotations

import colorsys
import re
from pathlib import Path

import numpy as np
from PIL import Image

VIEWS = ["back", "left", "front", "right"]                     # index = TinyHero folder
HAIR = ["black", "brown", "blonde", "red", "white", "grey", "green", "blue", "purple"]
OUTFIT = ["black", "brown", "yellow", "red", "white", "grey", "green", "blue", "purple"]
ATTRIBUTES = {"view": VIEWS, "hair": HAIR, "outfit": OUTFIT}

# region boxes (y0, y1, x0, x1) in the 64x64 frame, measured from the dataset
HAIR_BOX = (9, 18, 22, 42)
OUTFIT_BOX = (33, 44, 24, 40)


def _pixel_colour(r: float, g: float, b: float) -> str | None:
    h, s, v = colorsys.rgb_to_hsv(r, g, b)
    h *= 360
    if v < 0.18:
        return None                       # outline / shadow pixel: ignore
    if v < 0.32:
        return "black"
    if s < 0.22:
        return "white" if v > 0.78 else "grey"
    if h < 18 or h >= 335:
        return "red"
    if h < 45:
        return "brown" if v < 0.62 else "yellow"   # dark orange = brown, light = blonde/yellow
    if h < 70:
        return "yellow"
    if h < 170:
        return "green"
    if h < 225 or (h < 255 and s >= 0.5):
        return "blue"                     # saturated blues (hair dye, blue cloth)
    return "purple"                       # incl. the common desaturated lavender LPC shirt


def region_colour(rgba: np.ndarray, box: tuple[int, int, int, int]) -> str | None:
    y0, y1, x0, x1 = box
    reg = rgba[y0:y1, x0:x1]
    px = reg[reg[..., 3] > 0][:, :3] / 255.0
    names = [n for n in (_pixel_colour(*p) for p in px) if n is not None]
    if not names:
        return None
    vals, counts = np.unique(names, return_counts=True)
    return str(vals[np.argmax(counts)])


def label_sprite(path: str | Path) -> dict:
    rgba = np.asarray(Image.open(path).convert("RGBA"))
    hair = region_colour(rgba, HAIR_BOX) or "black"
    outfit = region_colour(rgba, OUTFIT_BOX) or "black"
    hair = {"yellow": "blonde"}.get(hair, hair)
    return {"hair": HAIR.index(hair), "outfit": OUTFIT.index(outfit)}


# --------------------------------------------------------------------------- prompt parsing
_SYNONYMS = {
    "view": {"back": ["back", "behind", "rear", "away"], "left": ["left"], "front": ["front", "facing", "forward", "toward"],
             "right": ["right"]},
    "colour": {"black": ["black", "dark", "raven", "ebony"], "brown": ["brown", "brunette", "chestnut", "leather"],
               "blonde": ["blonde", "blond", "golden", "gold", "yellow"], "yellow": ["yellow", "gold", "golden"],
               "red": ["red", "ginger", "crimson", "scarlet", "orange"], "white": ["white", "silver", "snow"],
               "grey": ["grey", "gray", "silver", "steel", "metal", "iron", "chainmail", "knight"],
               "green": ["green", "emerald", "forest"], "blue": ["blue", "azure", "navy", "cyan"],
               "purple": ["purple", "violet", "pink", "magenta", "lilac"]},
}
_HAIR_WORDS = ["hair", "haired", "head", "helmet", "helm", "hood", "hat", "headwear", "bald", "ponytail", "braid"]
_OUTFIT_WORDS = ["outfit", "shirt", "armour", "armor", "clothes", "clothing", "tunic", "robe", "dress", "chest",
                 "torso", "body", "wearing", "dressed", "vest", "top", "cloak", "suit", "chainmail"]


def _colour_of(word: str, allowed: list[str]) -> str | None:
    for name in allowed:
        if word in _SYNONYMS["colour"].get(name, []) or word == name:
            return name
    return None


_HAIR_ONLY = {"blonde", "blond", "ginger", "brunette", "raven", "bald"}


def parse_prompt(prompt: str) -> tuple[dict, list[str]]:
    """Map a free-text description to attribute labels (None = not specified -> random).

    The prompt is split into clauses at punctuation and at "and"/"with"; inside a
    clause each colour word attaches to the nearest hair noun (hair, helmet, hood...)
    or outfit noun (tunic, armour, dress...). Words like "blonde" always mean hair.
    Returns (labels, notes) where notes explain what was understood.
    """
    text = prompt.lower()
    words = re.findall(r"[a-z]+", text)
    labels: dict = {"view": None, "hair": None, "outfit": None}

    explicit = {"back": ["back", "behind", "rear", "away"], "left": ["left"], "right": ["right"],
                "front": ["front", "frontal"]}
    for w in words:
        for name, syns in explicit.items():
            if w in syns and labels["view"] is None:
                labels["view"] = VIEWS.index(name)
    if labels["view"] is None and any(w in ("facing", "forward", "toward", "towards", "camera") for w in words):
        labels["view"] = VIEWS.index("front")

    colour_words = {w for syns in _SYNONYMS["colour"].values() for w in syns} | set(_SYNONYMS["colour"])
    for clause in re.split(r"[,;.!?\n]|\band\b|\bwith\b", text):
        cw = re.findall(r"[a-z]+", clause)
        nouns = [(j, "hair") for j, w in enumerate(cw) if w in _HAIR_WORDS] + \
                [(j, "outfit") for j, w in enumerate(cw) if w in _OUTFIT_WORDS]
        for i, w in enumerate(cw):
            if w not in colour_words:
                continue
            if w in _HAIR_ONLY:
                target = "hair"
            elif nouns:   # nearest noun in the clause; ties go to the following noun ("red tunic")
                target = min(nouns, key=lambda n: (abs(n[0] - i), n[0] < i))[1]
            else:
                target = "hair" if labels["hair"] is None else "outfit"
            vocab = HAIR if target == "hair" else OUTFIT
            c = _colour_of(w, vocab)
            if c is not None and labels[target] is None:
                labels[target] = vocab.index(c)

    if labels["hair"] is None and any(w in ("helmet", "helm", "knight") for w in words):
        labels["hair"] = HAIR.index("grey")            # metal headwear is labelled grey in the data
    if labels["outfit"] is None and any(w in ("chainmail", "armour", "armor", "knight") for w in words):
        labels["outfit"] = OUTFIT.index("grey")
    notes = [f"{k}: {vocab[labels[k]]}" if labels[k] is not None else f"{k}: (not specified - random)"
             for k, vocab in ATTRIBUTES.items()]
    return labels, notes


# --------------------------------------------------------------------------- dataset labels + generated images
def label_image(img: Image.Image, background=(0, 0, 0), tol: int = 24) -> dict:
    """Label an RGB image (e.g. a generated sprite): pixels far from the background count as opaque."""
    rgb = np.asarray(img.convert("RGB"))
    alpha = (np.abs(rgb.astype(np.int16) - np.array(background, dtype=np.int16)).max(axis=2) > tol) * 255
    rgba = np.dstack([rgb, alpha.astype(np.uint8)])
    hair = region_colour(rgba, HAIR_BOX) or "black"
    outfit = region_colour(rgba, OUTFIT_BOX) or "black"
    hair = {"yellow": "blonde"}.get(hair, hair)
    return {"hair": HAIR.index(hair), "outfit": OUTFIT.index(outfit)}


def dataset_labels(root: str | Path, items: list[tuple[Path, int]]) -> np.ndarray:
    """(N, 3) int array [view, hair, outfit]; cached in <root>/../tinyhero_attributes.csv for inspection."""
    import csv
    root = Path(root)
    cache = root.parent / "tinyhero_attributes.csv"
    rows = {}
    if cache.exists():
        with open(cache, newline="") as f:
            rows = {r["file"]: r for r in csv.DictReader(f)}
    out, changed = [], False
    for path, view in items:
        key = f"{path.parent.name}/{path.name}"
        r = rows.get(key)
        if r is None:
            lab = label_sprite(path)
            r = rows[key] = {"file": key, "view": VIEWS[view], "hair": HAIR[lab["hair"]],
                             "outfit": OUTFIT[lab["outfit"]]}
            changed = True
        out.append([VIEWS.index(r["view"]), HAIR.index(r["hair"]), OUTFIT.index(r["outfit"])])
    if changed:  # merge: never drop rows labelled earlier (e.g. a smoke-test subset then the full set)
        with open(cache, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=["file", "view", "hair", "outfit"])
            w.writeheader()
            w.writerows(sorted(rows.values(), key=lambda r: r["file"]))
    return np.array(out, dtype=np.int64)

In [ ]:
attributes = fresh("attributes")
labels = attributes.dataset_labels(root, dataset.list_images(root))
for col, (name, vocab) in enumerate(attributes.ATTRIBUTES.items()):
    counts = np.bincount(labels[:, col], minlength=len(vocab))
    print(f"{name:7s}", ", ".join(f"{v}={c}" for v, c in zip(vocab, counts)))
# visual check: 8 real sprites per hair colour (rows)
items = dataset.list_images(root)
rng = np.random.default_rng(0); rows = []
for h in range(len(attributes.HAIR)):
    idx = np.where(labels[:, 1] == h)[0]
    rows += [dataset.load_rgb(items[i][0]) for i in rng.choice(idx, 8, replace=len(idx) < 8)]
print("Rows = hair labels:", ", ".join(attributes.HAIR))
display(utils.make_grid(rows, nrow=8, scale=2))
for p in ["a front-facing hero with green hair and a red tunic", "knight in armour seen from behind",
          "blonde girl wearing a blue dress, facing left"]:
    print(f"{p!r:55} -> {attributes.parse_prompt(p)[1]}")

## 9. GPU setup

Device priority: **CUDA** (Colab T4) → **MPS** (Apple Silicon) → **CPU**.
If no CUDA GPU is available, full training would take far too long, so the notebook **automatically switches to
smoke-test mode** (2 epochs, 256 images). This verifies the code path but does **not** produce a good model.
To fix it in Colab: `Runtime → Change runtime type → T4 GPU`, then *Run all* again.

In [ ]:
import torch
device = config.get_device()
print("PyTorch", torch.__version__, "| device:", config.describe_device(device))
if device.type == "cuda":
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"])

if not SMOKE_TEST and device.type != "cuda" and not ALLOW_FULL_TRAINING_WITHOUT_CUDA:
    print("\n" + "!" * 78)
    print("No CUDA GPU detected -> switching to SMOKE-TEST mode (2 epochs on 256 images).")
    print("The pipeline will be verified, but the resulting model is NOT fully trained.")
    print("In Colab: Runtime > Change runtime type > T4 GPU, then Run all.")
    print("!" * 78)
    SMOKE_TEST = True

if SMOKE_TEST:
    cfg = config.smoke_test_config(nz=LATENT_DIM, lr=LEARNING_RATE, seed=SEED, views=tuple(VIEWS),
                                   label_smoothing=LABEL_SMOOTHING, conditional=CONDITIONAL, d_cond=D_COND,
                                   ema_decay=EMA_DECAY)
else:
    cfg = config.Config(epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE, nz=LATENT_DIM,
                        checkpoint_every=CHECKPOINT_EVERY, sample_every=SAMPLE_EVERY, seed=SEED,
                        views=tuple(VIEWS), label_smoothing=LABEL_SMOOTHING, conditional=CONDITIONAL,
                        d_cond=D_COND, ema_decay=EMA_DECAY)
config.seed_everything(cfg.seed)
MODE = "SMOKE TEST" if SMOKE_TEST else "FULL TRAINING"
print(f"\nMode: {MODE} | epochs={cfg.epochs} batch={cfg.batch_size} subset={cfg.subset}")
print("checkpoints ->", cfg.checkpoint_dir, "\noutputs     ->", cfg.output_dir)

## 10. Generator implementation

The generator upsamples a 100-d noise vector into a 64×64 RGB sprite. A `ConvTranspose2d` with kernel 4,
stride 2, padding 1 exactly doubles the spatial size: $H_{out} = (H_{in}-1)\cdot 2 - 2 + 4 = 2H_{in}$.
The first layer (stride 1, padding 0) turns the 1×1 input into a 4×4 feature map.
`weights_init` applies the DCGAN initialisation; `sample_latent` draws $z$ on the CPU with a seeded
generator so the **same seed gives the same characters on CUDA, MPS and CPU**.

The **conditional generator** (`ConditionalGenerator`, next cell) reuses exactly this stack. Each label
(view, hair, outfit) is turned into a learned 16-d embedding and concatenated with $z$, so the first layer
sees $100 + 3 \times 16 = 148$ inputs.

In [ ]:
%%writefile models.py
"""DCGAN Generator and Discriminator for 64x64 RGB sprites (Radford et al., 2016).

Generator:      z (B,100,1,1) -> 4x4 -> 8x8 -> 16x16 -> 32x32 -> 64x64 RGB (Tanh)
Discriminator:  64x64 RGB -> 32 -> 16 -> 8 -> 4 -> one raw logit per image
The discriminator has NO sigmoid: we train with BCEWithLogitsLoss, which fuses
sigmoid + binary cross-entropy in a numerically stable way.
"""
from __future__ import annotations

import torch
import torch.nn as nn


def weights_init(m: nn.Module) -> None:
    """DCGAN initialisation: conv weights ~ N(0, 0.02), BN gamma ~ N(1, 0.02), beta = 0."""
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Embedding):
        nn.init.normal_(m.weight, 0.0, 0.02)


class Generator(nn.Module):
    def __init__(self, nz: int = 100, ngf: int = 64, nc: int = 3):
        super().__init__()
        self.nz = nz

        def up(cin, cout):
            # kernel 4, stride 2, padding 1 doubles the spatial size exactly
            return [nn.ConvTranspose2d(cin, cout, 4, 2, 1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(True)]

        self.net = nn.Sequential(
            # (nz,1,1) -> (ngf*8, 4, 4)
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            *up(ngf * 8, ngf * 4),   # -> 8x8
            *up(ngf * 4, ngf * 2),   # -> 16x16
            *up(ngf * 2, ngf),       # -> 32x32
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),  # -> 64x64
            nn.Tanh(),               # outputs in [-1, 1], matching normalised data
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        if z.dim() == 2:
            z = z[:, :, None, None]
        return self.net(z)


# ---- notebook-split ----

## 11. Discriminator implementation

The discriminator mirrors the generator with strided `Conv2d` layers and **LeakyReLU(0.2)** (so gradients still
flow for negative activations). There is **no BatchNorm on the first layer** and **no sigmoid at the end**:
it returns a raw logit for `BCEWithLogitsLoss`.

**Conditional discriminator.** `ConcatDiscriminator` appends the labels to the image as 22 one-hot
"label maps" (4 + 9 + 9 channels, constant over the image). D then judges *"is this a real sprite with these
attributes?"*, so the generator is rewarded only when it follows the labels. A projection discriminator
(Miyato & Koyama, 2018) is also implemented. In our experiments it overpowered the generator on this small
dataset (samples collapsed by epoch 6), so concatenation is the default. This cell also adds the latent
interpolation helpers (`lerp`, `slerp`).

In [ ]:
%%writefile -a models.py


class MinibatchStdDev(nn.Module):
    """Appends one feature map holding the average std-dev of features across the batch (Karras et al., 2018).

    Real batches are varied, a collapsed generator's batch is not, so D can detect and punish mode collapse.
    """

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        s = x.std(dim=0, unbiased=False).mean() if x.size(0) > 1 else x.new_zeros(())
        return torch.cat([x, s.expand(x.size(0), 1, x.size(2), x.size(3))], dim=1)


class Discriminator(nn.Module):
    """spectral_norm=True gives the SN-DCGAN variant (Miyato et al., 2018): every conv weight is divided by
    its largest singular value, limiting how sharply D can react to small input changes. BatchNorm is then
    replaced by Identity, the usual pairing."""

    def __init__(self, ndf: int = 64, nc: int = 3, dropout: float = 0.0, spectral_norm: bool = False,
                 minibatch_std: bool = False):
        super().__init__()
        sn = nn.utils.parametrizations.spectral_norm if spectral_norm else (lambda m: m)

        def down(cin, cout):
            norm = nn.Identity() if spectral_norm else nn.BatchNorm2d(cout)
            layers = [sn(nn.Conv2d(cin, cout, 4, 2, 1, bias=False)), norm, nn.LeakyReLU(0.2, inplace=True)]
            return layers + [nn.Dropout2d(dropout)]  # always present (p=0 is identity) -> stable state_dict keys

        self.net = nn.Sequential(
            # no BatchNorm on the input layer (as recommended in the DCGAN paper)
            sn(nn.Conv2d(nc, ndf, 4, 2, 1, bias=False)),   # 64 -> 32
            nn.LeakyReLU(0.2, inplace=True),
            *down(ndf, ndf * 2),                           # -> 16
            *down(ndf * 2, ndf * 4),                       # -> 8
            *down(ndf * 4, ndf * 8),                       # -> 4
            MinibatchStdDev() if minibatch_std else nn.Identity(),
            sn(nn.Conv2d(ndf * 8 + int(minibatch_std), 1, 4, 1, 0, bias=False)),  # -> 1x1 raw logit (no sigmoid!)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).view(-1)


class ConditionalGenerator(nn.Module):
    """Conditional DCGAN generator: G(z, y).

    Each attribute (view, hair colour, outfit colour) has a learned embedding; the
    embeddings are concatenated with z and fed to the same upsampling stack as the
    unconditional Generator. y is a (B, n_attributes) LongTensor of class indices.
    """

    def __init__(self, nz: int = 100, ngf: int = 64, nc: int = 3, n_classes=(4, 9, 9), emb_dim: int = 16):
        super().__init__()
        self.nz, self.n_classes, self.conditional = nz, tuple(n_classes), True
        self.embeds = nn.ModuleList(nn.Embedding(n, emb_dim) for n in self.n_classes)
        self.net = Generator(nz + emb_dim * len(self.n_classes), ngf, nc).net

    def forward(self, z: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        e = torch.cat([emb(y[:, i]) for i, emb in enumerate(self.embeds)], dim=1)
        return self.net(torch.cat([z.flatten(1), e], dim=1)[:, :, None, None])


class ProjectionDiscriminator(nn.Module):
    """Projection discriminator (Miyato & Koyama, 2018) on the DCGAN body:

        D(x, y) = w . phi(x)  +  sum_a  embed_a(y_a) . phi(x)

    phi(x) are the 4x4 features (summed spatially). The projection term is large only
    when the image matches its labels, so the generator is pushed to obey them.
    """

    def __init__(self, ndf: int = 64, nc: int = 3, n_classes=(4, 9, 9), dropout: float = 0.0,
                 spectral_norm: bool = False, minibatch_std: bool = False, pool: str = "mean"):
        super().__init__()
        base = Discriminator(ndf, nc, dropout=dropout, spectral_norm=spectral_norm, minibatch_std=minibatch_std)
        self.n_classes, self.conditional, self.pool = tuple(n_classes), True, pool
        self.features = nn.Sequential(*list(base.net)[:-1])
        self.out = base.net[-1]                     # unconditional 4x4 conv -> 1 logit
        feat = ndf * 8 + int(minibatch_std)
        self.embeds = nn.ModuleList(nn.Embedding(n, feat) for n in self.n_classes)

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        h = self.features(x)
        out = self.out(h).view(-1)
        phi = h.mean(dim=(2, 3)) if self.pool == "mean" else h.sum(dim=(2, 3))
        for i, emb in enumerate(self.embeds):
            out = out + (emb(y[:, i]) * phi).sum(dim=1)
        return out


class ConcatDiscriminator(nn.Module):
    """Classic conditional GAN discriminator (Mirza & Osindero, 2014): one-hot label maps are
    concatenated to the image as extra input channels, then the usual DCGAN body scores the pair."""

    def __init__(self, ndf: int = 64, nc: int = 3, n_classes=(4, 9, 9), dropout: float = 0.0,
                 spectral_norm: bool = False, minibatch_std: bool = False):
        super().__init__()
        self.n_classes, self.conditional = tuple(n_classes), True
        self.body = Discriminator(ndf, nc + sum(self.n_classes), dropout=dropout, spectral_norm=spectral_norm,
                                  minibatch_std=minibatch_std)

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        maps = [nn.functional.one_hot(y[:, i], n).float() for i, n in enumerate(self.n_classes)]
        m = torch.cat(maps, dim=1)[:, :, None, None].expand(-1, -1, x.size(2), x.size(3))
        return self.body(torch.cat([x, m], dim=1))


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def build_models(nz: int = 100, ngf: int = 64, ndf: int = 64, device: torch.device | str = "cpu",
                 d_dropout: float = 0.0, d_spectral_norm: bool = False, d_minibatch_std: bool = False,
                 n_classes: tuple | None = None, d_cond: str = "concat"):
    """n_classes=None -> unconditional DCGAN; e.g. (4, 9, 9) -> conditional G(z, y) and projection D(x, y)."""
    dkw = dict(dropout=d_dropout, spectral_norm=d_spectral_norm, minibatch_std=d_minibatch_std)
    if n_classes:
        G = ConditionalGenerator(nz, ngf, n_classes=n_classes).to(device)
        Dcls = {"projection": ProjectionDiscriminator, "concat": ConcatDiscriminator}[d_cond]
        D = Dcls(ndf, n_classes=n_classes, **dkw).to(device)
    else:
        G = Generator(nz, ngf).to(device)
        D = Discriminator(ndf, **dkw).to(device)
    G.apply(weights_init)
    D.apply(weights_init)
    return G, D


def sample_latent(n: int, nz: int = 100, seed: int | None = None, device: torch.device | str = "cpu") -> torch.Tensor:
    """Sample z ~ N(0, I) on the CPU with an explicit generator, then move to `device`.

    Sampling on the CPU means the same seed gives the same latent vectors on
    CUDA, MPS and CPU, so seeds are portable between Colab and the Mac.
    """
    g = torch.Generator(device="cpu")
    if seed is not None:
        g.manual_seed(int(seed))
    else:
        g.seed()
    return torch.randn(n, nz, 1, 1, generator=g).to(device)


def lerp(z1: torch.Tensor, z2: torch.Tensor, steps: int) -> torch.Tensor:
    """Linear interpolation z(t) = (1 - t) z1 + t z2 for t in [0, 1]."""
    t = torch.linspace(0, 1, steps, device=z1.device).view(-1, *([1] * (z1.dim())))
    return (1 - t) * z1.unsqueeze(0) + t * z2.unsqueeze(0)


def slerp(z1: torch.Tensor, z2: torch.Tensor, steps: int) -> torch.Tensor:
    """Spherical interpolation: keeps intermediate vectors at a 'typical' Gaussian norm."""
    a, b = z1.flatten(), z2.flatten()
    omega = torch.acos(torch.clamp(torch.dot(a / a.norm(), b / b.norm()), -1.0, 1.0))
    if omega.abs() < 1e-6:
        return lerp(z1, z2, steps)
    t = torch.linspace(0, 1, steps, device=z1.device)
    so = torch.sin(omega)
    out = [(torch.sin((1 - ti) * omega) / so) * a + (torch.sin(ti * omega) / so) * b for ti in t]
    return torch.stack(out).view(steps, *z1.shape)


def interpolate_latents(z1: torch.Tensor, z2: torch.Tensor, steps: int = 10, mode: str = "linear") -> torch.Tensor:
    if mode == "linear":
        return lerp(z1, z2, steps)
    if mode == "slerp":
        return slerp(z1, z2, steps)
    raise ValueError(f"Unknown interpolation mode {mode!r}")

## 12. Model architecture

Build both networks, apply DCGAN initialisation and trace the tensor shapes layer by layer.

In [ ]:
models = fresh("models")
G, D = models.build_models(cfg.nz, cfg.ngf, cfg.ndf, device,
                            n_classes=tuple(cfg.n_classes) if cfg.conditional else None, d_cond=cfg.d_cond)
print(G); print(D)

def trace(seq, x):
    print(f"{'input':<18} {tuple(x.shape[1:])}")
    for m in seq:
        x = m(x)
        if isinstance(m, (torch.nn.ConvTranspose2d, torch.nn.Conv2d)):
            print(f"{m.__class__.__name__:<18} {tuple(x.shape[1:])}")
    return x

with torch.no_grad():
    g_in = G.net[0].in_channels  # nz (+ label embeddings if conditional)
    print(f"\nGenerator shape trace (input = z{' + label embeddings' if cfg.conditional else ''})")
    fake = trace(G.net, torch.zeros(1, g_in, 1, 1, device=device))
    d_body = D.body.net if hasattr(D, "body") else (D.features if hasattr(D, "features") else D.net)
    d_in = d_body[0].in_channels
    print(f"\nDiscriminator shape trace (input = image{' + label maps' if d_in > 3 else ''})")
    trace(d_body, torch.cat([fake, torch.zeros(1, d_in - 3, 64, 64, device=device)], 1))
pG, pD = models.count_parameters(G), models.count_parameters(D)
print(f"\nGenerator parameters:     {pG:,}  (~{pG*4/1e6:.1f} MB fp32)")
print(f"Discriminator parameters: {pD:,}  (~{pD*4/1e6:.1f} MB fp32)")

## 13. Loss functions

`BCEWithLogitsLoss` combines the sigmoid and binary cross-entropy in one numerically stable operation
(log-sum-exp trick), which is why the discriminator outputs raw logits.

* **Discriminator:** real images → target **0.9** (one-sided label smoothing, Salimans et al. 2016 — stops D
  from becoming over-confident), generated images → target **0**.
* **Generator:** generated images → target **1** (non-saturating loss: "make D call my images real").

In [ ]:
crit = torch.nn.BCEWithLogitsLoss()
logits = torch.tensor([3.0, -2.0, 0.0])
targets = torch.tensor([0.9, 0.0, 1.0])
manual = -(targets * torch.log(torch.sigmoid(logits)) + (1 - targets) * torch.log(1 - torch.sigmoid(logits))).mean()
print(f"BCEWithLogitsLoss = {crit(logits, targets):.6f} | manual sigmoid+BCE = {manual:.6f}")
print("At the theoretical equilibrium D outputs 0.5 everywhere: loss_D = 2*ln2 = %.3f, loss_G = ln2 = %.3f"
      % (2 * np.log(2), np.log(2)))

## 14. Training loop

For every batch:

1. Load real sprites, sample $z\sim\mathcal{N}(0,I)$ and generate fakes $G(z)$.
2. **Update D** on the real batch and on `fake.detach()` — detaching stops the D loss from changing G.
3. **Update G** by passing the (non-detached) fakes through D with target "real".
4. Log both losses and $D(x)$, $D(G(z))$; abort with a clear error if a loss becomes NaN/Inf.

Every epoch, `latest.pt` (G, D, both Adam states, epoch, config, history, fixed noise, RNG) is saved, so a Colab
disconnect loses at most one epoch — re-running the notebook with `RESUME = True` continues where it stopped
**without resetting the optimizer state**. Numbered checkpoints are saved every `CHECKPOINT_EVERY` epochs, and a
grid of 64 fixed latent vectors is rendered every `SAMPLE_EVERY` epochs. Checkpoints contain only
`state_dict`s and plain Python types, so they load with `weights_only=True` on any device.

In [ ]:
%%writefile train.py
"""DCGAN adversarial training for PixelForge.

Examples
    python train.py --smoke-test                 # 2 epochs on 256 images, verifies the pipeline
    python train.py --epochs 50                  # full training (default config)
    python train.py --epochs 100 --resume auto   # continue from checkpoints/latest.pt
"""
from __future__ import annotations

import argparse
import copy
import csv
import math
import os
import shutil
import time
from pathlib import Path

import torch
import torch.nn as nn
from tqdm.auto import tqdm

from config import Config, describe_device, get_device, seed_everything, smoke_test_config
from dataset import build_dataset, make_dataloader
from models import build_models, count_parameters, sample_latent
from utils import make_grid, tensor_to_pil

EPOCH_FIELDS = ["epoch", "loss_d", "loss_g", "d_real", "d_fake", "d_fake_after_g", "sample_diversity",
                "seconds", "iterations"]
COLLAPSE_RATIO = 0.25  # warn below 25% of real variety (collapsed probes: 9-22%; healthy black run: ~37%)
COLLAPSE_RATIO_COND = 0.10  # conditional: variety across z at ONE fixed label vs. real variety over all labels
ITER_FIELDS = ["epoch", "iteration", "loss_d", "loss_g", "d_real", "d_fake"]


# --------------------------------------------------------------------------- checkpoints
def save_checkpoint(path: str | Path, G: nn.Module, D: nn.Module, opt_g, opt_d, epoch: int,
                    cfg: Config, history: list[dict], fixed_noise: torch.Tensor, history_csv: str,
                    extra: dict | None = None) -> Path:
    """Portable checkpoint: state_dicts only (no pickled model objects), tensors moved to CPU."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ckpt = {
        "format": "pixelforge-dcgan-v1",
        "epoch": int(epoch),
        "config": cfg.to_dict(),
        "generator": {k: v.detach().cpu() for k, v in G.state_dict().items()},
        "discriminator": {k: v.detach().cpu() for k, v in D.state_dict().items()},
        "opt_g": _optimizer_state_to_cpu(opt_g.state_dict()),
        "opt_d": _optimizer_state_to_cpu(opt_d.state_dict()),
        "history": history,
        "history_csv": str(history_csv),
        "fixed_noise": fixed_noise.detach().cpu(),
        "params": {"generator": count_parameters(G), "discriminator": count_parameters(D)},
        "rng_cpu": torch.get_rng_state(),
        **(extra or {}),
    }
    tmp = path.with_suffix(".tmp")
    torch.save(ckpt, tmp)
    tmp.replace(path)  # atomic: an interrupted save never corrupts the previous file
    return path


def save_generator_only(path: str | Path, G: nn.Module, cfg: Config, epoch: int, history: list[dict],
                        ema: bool = False) -> Path:
    """Small (~14 MB) inference-only file for generate.py and the Streamlit app (EMA weights if available)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "format": "pixelforge-generator-v1",
        "epoch": int(epoch),
        "config": cfg.to_dict(),
        "generator": {k: v.detach().cpu() for k, v in G.state_dict().items()},
        "params": {"generator": count_parameters(G)},
        "history": history,
        "ema": ema,
    }, path)
    return path


def _optimizer_state_to_cpu(sd: dict) -> dict:
    out = {"param_groups": sd["param_groups"], "state": {}}
    for k, st in sd["state"].items():
        out["state"][k] = {n: (v.detach().cpu() if torch.is_tensor(v) else v) for n, v in st.items()}
    return out


def load_checkpoint(path: str | Path) -> dict:
    """Always load onto the CPU first so CUDA-trained files open on MPS/CPU and vice versa."""
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:  # very old torch without weights_only
        return torch.load(path, map_location="cpu")


def find_latest_checkpoint(checkpoint_dir: str | Path) -> Path | None:
    d = Path(checkpoint_dir)
    if (d / "latest.pt").exists():
        return d / "latest.pt"
    eps = sorted(d.glob("pixelforge_epoch_*.pt"))
    return eps[-1] if eps else None


def _prune(checkpoint_dir: Path, keep_last: int) -> None:
    if keep_last <= 0:
        return
    eps = sorted(checkpoint_dir.glob("pixelforge_epoch_*.pt"))
    for old in eps[:-keep_last]:
        old.unlink(missing_ok=True)


# --------------------------------------------------------------------------- csv helpers
def _rewrite_csv(path: Path, fields: list[str], rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)


def _append_csv(path: Path, fields: list[str], rows: list[dict]) -> None:
    new = not path.exists()
    with open(path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        if new:
            w.writeheader()
        w.writerows(rows)


def _truncate_iter_csv(path: Path, last_epoch: int) -> None:
    """On resume, drop iteration rows logged after the checkpoint we resume from."""
    if not path.exists():
        return
    with open(path, newline="") as f:
        rows = [r for r in csv.DictReader(f) if int(r["epoch"]) <= last_epoch]
    _rewrite_csv(path, ITER_FIELDS, rows)


@torch.no_grad()
def sample_diversity(G: nn.Module, fixed_noise: torch.Tensor, labels: torch.Tensor | None = None) -> float:
    """Mean per-pixel std across the fixed-noise samples (in [0,1] units). ~0 => all z give the same image.

    For a conditional G pass ONE label combination repeated, so the measure reflects variety across z only.
    """
    was_training = G.training
    G.eval()
    x = (_g(G, fixed_noise, labels).float() + 1) / 2
    if was_training:
        G.train()
    return float(x.std(dim=0).mean())


def _g(G: nn.Module, z: torch.Tensor, y: torch.Tensor | None):
    return G(z, y) if getattr(G, "conditional", False) else G(z)


def _d(D: nn.Module, x: torch.Tensor, y: torch.Tensor | None):
    return D(x, y) if getattr(D, "conditional", False) else D(x)


def fixed_condition_grid(n: int, device) -> torch.Tensor:
    """Labels for the 8x8 monitoring grid: row = hair colour, column = view (back/left/front/right x2),
    outfit cycles per column. Makes it easy to see whether G obeys the labels."""
    from attributes import HAIR, OUTFIT
    hair_rows = [HAIR.index(h) for h in ("black", "brown", "blonde", "red", "grey", "green", "blue", "purple")]
    outfits = [OUTFIT.index(o) for o in ("purple", "grey", "yellow", "blue", "red", "green", "brown", "white")]
    y = [[c % 4, hair_rows[(k // 8) % 8], outfits[c]] for k in range(n) for c in [k % 8]]
    return torch.tensor(y, dtype=torch.long, device=device)


def real_diversity(data_uint8: torch.Tensor, n: int = 64, seed: int = 0) -> float:
    """Same measure as sample_diversity() on n random real images: the reference for collapse warnings."""
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(data_uint8.shape[0], generator=g)[:n]
    return float((data_uint8[idx].float() / 255).std(dim=0).mean())


@torch.no_grad()
def save_progress_grid(G: nn.Module, fixed_noise: torch.Tensor, path: Path, nrow: int = 8, scale: int = 2,
                       labels: torch.Tensor | None = None) -> Path:
    was_training = G.training
    G.eval()
    imgs = tensor_to_pil(_g(G, fixed_noise, labels))
    if was_training:
        G.train()
    path.parent.mkdir(parents=True, exist_ok=True)
    make_grid(imgs, nrow=nrow, scale=scale).save(path)
    return path


# --------------------------------------------------------------------------- training
def train(cfg: Config, resume: str | None = None, show_progress: bool = True, on_epoch_end=None) -> dict:
    """Run adversarial training. Returns a summary dict with paths and the per-epoch history.

    resume: None | "auto" (checkpoint_dir/latest.pt if present) | path to a checkpoint.
    on_epoch_end(epoch, G, fixed_noise, history): optional callback (used by the notebook).
    """
    device = get_device(cfg.device)
    seed_everything(cfg.seed)
    ckpt_dir, out_dir = Path(cfg.checkpoint_dir), Path(cfg.output_dir)
    progress_dir = out_dir / "training_progress"
    for d in (ckpt_dir, out_dir, progress_dir, out_dir / "plots"):
        d.mkdir(parents=True, exist_ok=True)
    history_csv, iter_csv = out_dir / "training_history.csv", out_dir / "training_iterations.csv"

    ds = build_dataset(cfg)
    loader = make_dataloader(ds, cfg.batch_size, shuffle=True, num_workers=cfg.num_workers,
                             seed=cfg.seed, pin_memory=device.type == "cuda")
    if len(loader) == 0:
        raise ValueError(f"Dataset has {len(ds)} images, fewer than batch_size={cfg.batch_size}.")

    resume_path = find_latest_checkpoint(ckpt_dir) if resume == "auto" else (Path(resume) if resume else None)
    ck = load_checkpoint(resume_path) if resume_path is not None and resume_path.exists() else None
    if ck is not None:  # architecture must match the checkpoint, whatever the CLI says
        for k in ("nz", "ngf", "ndf", "d_dropout", "d_spectral_norm", "d_minibatch_std", "conditional", "n_classes", "d_cond"):
            if k in ck["config"]:
                setattr(cfg, k, tuple(ck["config"][k]) if k == "n_classes" else ck["config"][k])
            elif k in ("d_spectral_norm", "d_minibatch_std", "conditional"):
                setattr(cfg, k, False)  # checkpoints made before these options existed

    if cfg.conditional and cfg.views != (0, 1, 2, 3):
        print("Note: conditional model trained on a subset of views; unseen views will not be learned.")
    G, D = build_models(cfg.nz, cfg.ngf, cfg.ndf, device, d_dropout=cfg.d_dropout,
                        d_spectral_norm=cfg.d_spectral_norm, d_minibatch_std=cfg.d_minibatch_std,
                        n_classes=tuple(cfg.n_classes) if cfg.conditional else None, d_cond=cfg.d_cond)
    G_ema = None
    if cfg.ema_decay > 0:  # Polyak/EMA average of generator weights, used for all sampling
        G_ema = copy.deepcopy(G).eval()
        for p in G_ema.parameters():
            p.requires_grad_(False)
    fixed_labels = fixed_condition_grid(cfg.num_fixed_samples, device) if cfg.conditional else None
    div_labels = fixed_labels[:1].repeat(cfg.num_fixed_samples, 1) if cfg.conditional else None
    G_sample = lambda: G_ema if G_ema is not None else G  # noqa: E731
    opt_g = torch.optim.Adam(G.parameters(), lr=cfg.lr, betas=(cfg.beta1, cfg.beta2))
    opt_d = torch.optim.Adam(D.parameters(), lr=cfg.lr, betas=(cfg.beta1, cfg.beta2))
    criterion = nn.BCEWithLogitsLoss()
    fixed_noise = sample_latent(cfg.num_fixed_samples, cfg.nz, seed=cfg.seed, device=device)

    start_epoch, history = 0, []
    if ck is not None:
        if "discriminator" not in ck:
            raise ValueError(f"{resume_path} is a generator-only file; resume needs a full checkpoint.")
        G.load_state_dict(ck["generator"])
        D.load_state_dict(ck["discriminator"])
        if G_ema is not None:
            G_ema.load_state_dict(ck.get("generator_ema", ck["generator"]))
        opt_g.load_state_dict(ck["opt_g"])   # optimizer moments are restored, not reset
        opt_d.load_state_dict(ck["opt_d"])
        start_epoch, history = int(ck["epoch"]), list(ck.get("history", []))
        fixed_noise = ck["fixed_noise"].to(device)
        torch.set_rng_state(ck["rng_cpu"])
        _rewrite_csv(history_csv, EPOCH_FIELDS, history)
        _truncate_iter_csv(iter_csv, start_epoch)
        print(f"Resumed from {resume_path} at epoch {start_epoch}")
    elif resume and resume != "auto":
        raise FileNotFoundError(f"Checkpoint not found: {resume}")
    else:
        for p in (history_csv, iter_csv):
            p.unlink(missing_ok=True)
        save_progress_grid(G_sample(), fixed_noise, progress_dir / "epoch_000.png", labels=fixed_labels)

    pg, pd_ = count_parameters(G), count_parameters(D)
    real_div = real_diversity(ds.data, n=cfg.num_fixed_samples)
    print(f"Device: {describe_device(device)}")
    print(f"Images: {len(ds)} | batches/epoch: {len(loader)} | batch size: {cfg.batch_size}")
    print(f"Generator params: {pg:,} | Discriminator params: {pd_:,}")
    print(f"Real-data diversity reference: {real_div:.4f} (collapse warning below {COLLAPSE_RATIO * real_div:.4f})")

    collapse_ratio = COLLAPSE_RATIO if not cfg.conditional else COLLAPSE_RATIO_COND
    ema_steps = 0

    @torch.no_grad()
    def ema_update():
        nonlocal ema_steps
        ema_steps += 1
        d = min(cfg.ema_decay, (1 + ema_steps) / (10 + ema_steps))  # warm-up: short memory early on
        for pe, p in zip(G_ema.parameters(), G.parameters()):
            pe.lerp_(p, 1 - d)
        for be, b in zip(G_ema.buffers(), G.buffers()):
            be.copy_(b)

    real_target_value = 1.0 - cfg.label_smoothing  # one-sided label smoothing (real labels only)
    last_epoch = start_epoch
    for epoch in range(start_epoch + 1, cfg.epochs + 1):
        G.train()
        D.train()
        t0 = time.time()
        sums = {"loss_d": 0.0, "loss_g": 0.0, "d_real": 0.0, "d_fake": 0.0, "d_fake_after_g": 0.0}
        iter_rows = []
        # instance noise (Sonderby et al. 2016): blur the boundary between real and fake for D, annealed to 0
        noise_std = cfg.instance_noise * max(0.0, 1.0 - (epoch - 1) / max(1, cfg.epochs))
        bar = tqdm(loader, desc=f"Epoch {epoch}/{cfg.epochs}", leave=False, disable=not show_progress)
        for i, (real, yb) in enumerate(bar):
            real = real.to(device, non_blocking=True)
            y = yb.to(device) if cfg.conditional else None  # fakes use the real batch's labels
            b = real.size(0)
            real_t = torch.full((b,), real_target_value, device=device)
            if cfg.real_label_jitter > 0:  # noisy labels (as in the original TinyHero DCGAN notebook)
                real_t = real_t + (torch.rand(b, device=device) * 2 - 1) * cfg.real_label_jitter
            fake_t = torch.rand(b, device=device) * cfg.fake_label_max if cfg.fake_label_max > 0 \
                else torch.zeros(b, device=device)

            # (1) Discriminator: maximise log D(x) + log(1 - D(G(z)))
            z = torch.randn(b, cfg.nz, 1, 1, device=device)
            fake = _g(G, z, y)
            opt_d.zero_grad(set_to_none=True)
            if noise_std > 0:
                d_real_in = real + noise_std * torch.randn_like(real)
                d_fake_in = fake + noise_std * torch.randn_like(fake)
            else:
                d_real_in, d_fake_in = real, fake
            out_real = _d(D, d_real_in, y)
            out_fake = _d(D, d_fake_in.detach(), y)  # detach: no generator gradients in the D step
            loss_d = criterion(out_real, real_t) + criterion(out_fake, fake_t)
            loss_d.backward()
            opt_d.step()

            # (2) Generator: non-saturating loss, maximise log D(G(z))
            opt_g.zero_grad(set_to_none=True)
            out_fake_g = _d(D, d_fake_in, y)
            loss_g = criterion(out_fake_g, torch.ones(b, device=device))
            loss_g.backward()
            opt_g.step()
            if G_ema is not None:
                ema_update()

            ld, lg = loss_d.item(), loss_g.item()
            if not (math.isfinite(ld) and math.isfinite(lg)):
                raise FloatingPointError(
                    f"Non-finite loss at epoch {epoch}, iteration {i} (loss_d={ld}, loss_g={lg}). "
                    "Try a lower --lr or resume from the last checkpoint.")
            dr = torch.sigmoid(out_real).mean().item()
            df = torch.sigmoid(out_fake).mean().item()
            dfg = torch.sigmoid(out_fake_g).mean().item()
            for k, v in zip(sums, (ld, lg, dr, df, dfg)):
                sums[k] += v
            iter_rows.append({"epoch": epoch, "iteration": (epoch - 1) * len(loader) + i,
                              "loss_d": round(ld, 5), "loss_g": round(lg, 5),
                              "d_real": round(dr, 5), "d_fake": round(df, 5)})
            bar.set_postfix(loss_d=f"{ld:.3f}", loss_g=f"{lg:.3f}", D_x=f"{dr:.2f}", D_Gz=f"{df:.2f}")

        n = len(loader)
        row = {k: round(v / n, 5) for k, v in sums.items()}
        row.update(epoch=epoch, sample_diversity=round(sample_diversity(G_sample(), fixed_noise, div_labels), 5),
                   seconds=round(time.time() - t0, 2), iterations=n)
        history.append(row)
        _append_csv(history_csv, EPOCH_FIELDS, [row])
        _append_csv(iter_csv, ITER_FIELDS, iter_rows)
        print(f"[{epoch:3d}/{cfg.epochs}] loss_D={row['loss_d']:.4f} loss_G={row['loss_g']:.4f} "
              f"D(x)={row['d_real']:.3f} D(G(z))={row['d_fake']:.3f} div={row['sample_diversity']:.3f} "
              f"({row['seconds']:.1f}s)", flush=True)
        if row["sample_diversity"] < collapse_ratio * real_div and epoch >= 3:
            print(f"WARNING: possible mode collapse - fixed-noise samples are nearly identical "
                  f"(diversity {row['sample_diversity']:.4f} < {collapse_ratio:.0%} of real {real_div:.4f}). "
                  "Consider --instance-noise / --d-dropout or a lower learning rate.", flush=True)

        if epoch % cfg.sample_every == 0 or epoch == cfg.epochs or epoch == 1:
            save_progress_grid(G_sample(), fixed_noise, progress_dir / f"epoch_{epoch:03d}.png", labels=fixed_labels)
        # latest.pt every epoch -> a Colab disconnect loses at most one epoch
        extra = {"fixed_labels": fixed_labels.cpu() if fixed_labels is not None else None}
        if G_ema is not None:
            extra["generator_ema"] = {k: v.detach().cpu() for k, v in G_ema.state_dict().items()}
        save_checkpoint(ckpt_dir / "latest.pt", G, D, opt_g, opt_d, epoch, cfg, history, fixed_noise,
                        str(history_csv), extra=extra)
        if epoch % cfg.checkpoint_every == 0:
            # identical content to latest.pt right now -> hard link (no extra disk), copy as fallback
            numbered = ckpt_dir / f"pixelforge_epoch_{epoch:03d}.pt"
            numbered.unlink(missing_ok=True)
            try:
                os.link(ckpt_dir / "latest.pt", numbered)
            except OSError:
                shutil.copy2(ckpt_dir / "latest.pt", numbered)
            _prune(ckpt_dir, cfg.keep_last)
        if on_epoch_end is not None:
            on_epoch_end(epoch, G, fixed_noise, history)
        last_epoch = epoch

    final = save_generator_only(ckpt_dir / "generator_final.pt", G_sample(), cfg, last_epoch, history,
                                ema=G_ema is not None)
    try:
        from evaluate import plot_losses
        plot_losses(history_csv, iter_csv, out_dir / "plots" / "losses.png")
    except ImportError:
        pass  # evaluate.py not present yet (notebook writes it in a later cell)
    except Exception as e:  # plotting must never lose a finished training run
        print(f"(loss plot skipped: {e})")
    print(f"Done. Generator weights: {final}")
    return {"generator_path": str(final), "latest_path": str(ckpt_dir / "latest.pt"), "history": history,
            "history_csv": str(history_csv), "device": str(device), "epochs_completed": last_epoch,
            "params": {"generator": pg, "discriminator": pd_}}


def parse_args(argv=None):
    d = Config()
    p = argparse.ArgumentParser(description="Train the PixelForge DCGAN on TinyHero")
    p.add_argument("--smoke-test", action="store_true", help="2 epochs on 256 images, outputs in */smoke/")
    p.add_argument("--epochs", type=int)
    p.add_argument("--batch-size", type=int)
    p.add_argument("--lr", type=float)
    p.add_argument("--beta1", type=float)
    p.add_argument("--beta2", type=float)
    p.add_argument("--nz", type=int, help="latent dimension")
    p.add_argument("--ngf", type=int)
    p.add_argument("--ndf", type=int)
    p.add_argument("--seed", type=int)
    p.add_argument("--subset", type=int, help="train on N random images only")
    p.add_argument("--views", type=int, nargs="+", choices=[0, 1, 2, 3],
                   help="0=back 1=left 2=front 3=right (default: all). --views 2 = front only (912 images)")
    p.add_argument("--hflip", action="store_true", help="random horizontal flip augmentation")
    p.add_argument("--unconditional", action="store_true",
                   help="plain DCGAN without attribute labels (no prompt control)")
    p.add_argument("--d-cond", choices=["projection", "concat"], help="how the discriminator sees labels")
    p.add_argument("--ema-decay", type=float, help="EMA of generator weights for sampling (0 = off, default 0.998)")
    p.add_argument("--label-smoothing", type=float, help=f"one-sided; 0 disables (default {d.label_smoothing})")
    p.add_argument("--real-label-jitter", type=float, help="real targets ~ U(r-j, r+j)")
    p.add_argument("--fake-label-max", type=float, help="fake targets ~ U(0, m)")
    p.add_argument("--background", type=int, nargs=3, metavar=("R", "G", "B"),
                   help="colour that transparent pixels are composited onto (default 0 0 0; 255 255 255 = white)")
    p.add_argument("--instance-noise", type=float, help="std of annealed Gaussian noise on D inputs (0 = off)")
    p.add_argument("--d-dropout", type=float, help="dropout probability in the discriminator (0 = off)")
    p.add_argument("--spectral-norm", dest="d_spectral_norm", action="store_true", default=None,
                   help="spectral normalisation in D (SN-DCGAN)")
    p.add_argument("--no-spectral-norm", dest="d_spectral_norm", action="store_false",
                   help="plain DCGAN discriminator with BatchNorm")
    p.add_argument("--minibatch-std", dest="d_minibatch_std", action="store_true", default=None,
                   help="minibatch std-dev feature in D (counters mode collapse)")
    p.add_argument("--no-minibatch-std", dest="d_minibatch_std", action="store_false")
    p.add_argument("--sample-every", type=int)
    p.add_argument("--checkpoint-every", type=int)
    p.add_argument("--keep-last", type=int, help="number of epoch checkpoints to keep (0 = all)")
    p.add_argument("--data-dir")
    p.add_argument("--checkpoint-dir")
    p.add_argument("--output-dir")
    p.add_argument("--device", choices=["auto", "cuda", "mps", "cpu"])
    p.add_argument("--num-workers", type=int)
    p.add_argument("--resume", nargs="?", const="auto", default=None,
                   help="resume from a checkpoint path, or no value / 'auto' for <checkpoint-dir>/latest.pt")
    p.add_argument("--no-progress", action="store_true", help="disable tqdm bars")
    return p.parse_args(argv)


def config_from_args(args) -> Config:
    cfg = smoke_test_config() if args.smoke_test else Config()
    for name in ("epochs", "batch_size", "lr", "beta1", "beta2", "nz", "ngf", "ndf", "seed", "subset",
                 "ema_decay", "d_cond", "label_smoothing", "real_label_jitter", "fake_label_max", "instance_noise", "d_dropout", "d_spectral_norm", "d_minibatch_std", "sample_every", "checkpoint_every", "keep_last", "data_dir",
                 "checkpoint_dir", "output_dir", "device", "num_workers"):
        v = getattr(args, name)
        if v is not None:
            setattr(cfg, name, v)
    if args.views:
        cfg.views = tuple(args.views)
    if args.background:
        cfg.background = tuple(args.background)
    if args.hflip:
        cfg.hflip = True
    if args.unconditional:
        cfg.conditional = False
    return cfg


if __name__ == "__main__":
    a = parse_args()
    train(config_from_args(a), resume=a.resume, show_progress=not a.no_progress)

In [ ]:
train = fresh("train")

def show_progress(epoch, G, fixed_noise, history):
    if epoch == 1 or epoch % cfg.sample_every == 0 or epoch == cfg.epochs:
        p = Path(cfg.output_dir) / "training_progress" / f"epoch_{epoch:03d}.png"
        if p.exists():
            print(f"Fixed-noise samples after epoch {epoch}:")
            display(IPImage(str(p), width=520))

result = train.train(cfg, resume="auto" if RESUME else None, on_epoch_end=show_progress)
print(f"\n{MODE} finished: {result['epochs_completed']} epochs on {result['device']}")

## 15. Generator and discriminator losses

`evaluate.py` holds all plotting and evaluation code (used again in Section 19).

**Why the loss alone cannot tell us sample quality:** the two losses measure a moving two-player game, not
distance to the data distribution. When G improves, D's task gets harder and its loss rises; when D improves,
G's loss rises — even if images are getting better. Losses can look stable while the generator mode-collapses.
We therefore always inspect samples, progression grids and diversity statistics as well.

In [ ]:
%%writefile evaluate.py
"""PixelForge evaluation: five lightweight, honest experiments.

  1. Training losses (G vs D)                    -> plots/losses.png
  2. Fixed-noise quality progression over epochs -> plots/progression.png
  3. Real vs generated side-by-side              -> plots/real_vs_generated.png
  4. Latent interpolation strips                 -> plots/interpolation_experiment.png
  5. Diversity + memorisation (nearest neighbours in pixel space)
                                                 -> plots/nearest_neighbors.png, plots/diversity.png
All numbers are computed from the actual checkpoint and dataset and written to
outputs/evaluation_metrics.json. Nothing is hard-coded.

    python evaluate.py                       # uses checkpoints/generator_final.pt
    python evaluate.py --checkpoint checkpoints/latest.pt --num-samples 1000
"""
from __future__ import annotations

import argparse
import csv
import json
import re
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from matplotlib.figure import Figure
from PIL import Image

from config import PROJECT_ROOT, Config, get_device

# Chart styling (validated categorical slots for a dark surface).
SURFACE, INK, INK_MUTED, GRID = "#14111f", "#f0efec", "#a9a6b8", "#2c2840"
C_GEN, C_DISC = "#3987e5", "#d95926"


def _style(ax, title=None, xlabel=None, ylabel=None):
    ax.set_facecolor(SURFACE)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK_MUTED, labelsize=9)
    ax.grid(True, color=GRID, linewidth=0.6, alpha=0.8)
    ax.set_axisbelow(True)
    if title:
        ax.set_title(title, color=INK, fontsize=11, loc="left", pad=10)
    if xlabel:
        ax.set_xlabel(xlabel, color=INK_MUTED, fontsize=9)
    if ylabel:
        ax.set_ylabel(ylabel, color=INK_MUTED, fontsize=9)


def _legend(ax):
    leg = ax.legend(frameon=False, fontsize=9, loc="upper right")
    for t in leg.get_texts():
        t.set_color(INK)


def _fig(w, h, nrows=1, ncols=1):
    fig = Figure(figsize=(w, h), dpi=120, facecolor=SURFACE)
    axes = fig.subplots(nrows, ncols, squeeze=False)
    return fig, axes


def _read_csv(path):
    with open(path, newline="") as f:
        return [{k: float(v) for k, v in r.items()} for r in csv.DictReader(f)]


def _smooth(y, k):
    if len(y) < k or k <= 1:
        return np.asarray(y)
    return np.convolve(y, np.ones(k) / k, mode="valid")


# --------------------------------------------------------------------------- Experiment 1
def plot_losses(history_csv, iter_csv=None, out_path="outputs/plots/losses.png") -> Path:
    """Generator and discriminator losses: one panel per loss (one axis each), plus D outputs."""
    hist = _read_csv(history_csv)
    it = _read_csv(iter_csv) if iter_csv and Path(iter_csv).exists() else []
    fig, ax = _fig(12, 3.6, 1, 3)
    ep = [r["epoch"] for r in hist]
    for a, key, color, name in ((ax[0, 0], "loss_g", C_GEN, "Generator loss"),
                                (ax[0, 1], "loss_d", C_DISC, "Discriminator loss")):
        if it:
            x = np.array([r["iteration"] for r in it])
            y = np.array([r[key] for r in it])
            k = max(1, len(y) // 60)
            if k > 1:
                a.plot(x, y, color=color, alpha=0.22, linewidth=1, label="per iteration")
                a.plot(x[k - 1:], _smooth(y, k), color=color, linewidth=2, label=f"moving avg ({k} iters)")
            else:
                a.plot(x, y, color=color, linewidth=2, label="per iteration")
            _style(a, name, "iteration", "BCE loss")
        else:
            a.plot(ep, [r[key] for r in hist], color=color, linewidth=2, marker="o", markersize=4, label="epoch mean")
            _style(a, name, "epoch", "BCE loss")
        _legend(a)
    a = ax[0, 2]
    a.plot(ep, [r["d_real"] for r in hist], color=C_GEN, linewidth=2, marker="o", markersize=4, label="D(x) real")
    a.plot(ep, [r["d_fake"] for r in hist], color=C_DISC, linewidth=2, marker="o", markersize=4, label="D(G(z)) fake")
    a.set_ylim(0, 1)
    _style(a, "Discriminator confidence", "epoch", "mean sigmoid output")
    _legend(a)
    fig.tight_layout()
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, facecolor=SURFACE)
    return out_path


# --------------------------------------------------------------------------- Experiment 2
def progression_montage(progress_dir, out_path, n_sprites: int = 8, max_rows: int = 12) -> Path | None:
    """First row of each fixed-noise grid, stacked by epoch: same z, increasing training."""
    files = sorted(Path(progress_dir).glob("epoch_*.png"))
    if not files:
        return None
    if len(files) > max_rows:  # keep first, last, and evenly spaced in between
        idx = np.unique(np.linspace(0, len(files) - 1, max_rows).round().astype(int))
        files = [files[i] for i in idx]
    rows = []
    for f in files:
        g = Image.open(f).convert("RGB")
        # grids are made with padding=2, 64px tiles, upscale s (inferred from width, 8 per row)
        s = g.width // (8 * 64 + 9 * 2)
        tile, pad = 64 * s, 2 * s
        rows.append((int(re.findall(r"\d+", f.stem)[0]),
                     [g.crop((pad + c * (tile + pad), pad, pad + c * (tile + pad) + tile, pad + tile)) for c in range(n_sprites)]))
    fig, ax = _fig(n_sprites * 0.9 + 1, len(rows) * 0.95 + 0.6, len(rows), n_sprites)
    for r, (epoch, tiles) in enumerate(rows):
        for c, t in enumerate(tiles):
            a = ax[r, c]
            a.imshow(t, interpolation="nearest")
            a.set_xticks([]), a.set_yticks([])
            for sp in a.spines.values():
                sp.set_visible(False)
            if c == 0:
                a.set_ylabel(f"epoch {epoch}", color=INK, fontsize=8, rotation=0, ha="right", va="center")
    fig.suptitle("Same latent vectors across training (fixed noise)", color=INK, fontsize=11, x=0.02, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.97))  # leave room for the suptitle
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, facecolor=SURFACE)
    return out_path


# --------------------------------------------------------------------------- helpers
def _imshow_grid(ax, images, title):
    from utils import make_grid
    ax.imshow(make_grid(images, nrow=int(np.ceil(np.sqrt(len(images))))), interpolation="nearest")
    ax.set_xticks([]), ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_title(title, color=INK, fontsize=12, pad=8)


def _to_pil(x01: torch.Tensor) -> list[Image.Image]:
    arr = (x01.clamp(0, 1).permute(0, 2, 3, 1).cpu().numpy() * 255).round().astype(np.uint8)
    return [Image.fromarray(a) for a in arr]


@torch.no_grad()
def generate_batch01(G, n, seed, device, batch=256, labels: torch.Tensor | None = None) -> torch.Tensor:
    """n generated images in [0,1]. Conditional G: `labels` (n,3) or, if None, random labels with `seed`."""
    from generate import is_conditional, make_labels
    from models import sample_latent
    z = sample_latent(n, G.nz, seed=seed, device="cpu")
    if is_conditional(G) and labels is None:
        labels = make_labels(G, n, None, seed, "cpu")
    outs = []
    for i in range(0, n, batch):
        zi = z[i:i + batch].to(device)
        x = G(zi, labels[i:i + batch].to(device)) if is_conditional(G) else G(zi)
        outs.append(((x.float().cpu() + 1) / 2).clamp(0, 1))
    return torch.cat(outs)


@torch.no_grad()
def controllability(G, device, background, per_class: int = 48, seed: int = 0) -> dict:
    """Experiment 6: does G obey the requested hair / outfit colour?

    For every class c of an attribute, generate `per_class` sprites asking for c (other attributes random),
    label them with the same automatic labeller used on the training data, and count matches.
    Baseline: the same labels compared against a *shuffled* copy of the requests (= a model that ignores them).
    """
    from attributes import HAIR, OUTFIT, label_image
    from generate import make_labels
    res = {}
    for col, name, vocab in ((1, "hair", HAIR), (2, "outfit", OUTFIT)):
        req, got = [], []
        for c in range(len(vocab)):
            y = make_labels(G, per_class, {name: c}, seed + 31 * c, "cpu")
            imgs = _to_pil(generate_batch01(G, per_class, seed + 31 * c + 1, device, labels=y))
            req += [c] * per_class
            got += [label_image(im, background)[name] for im in imgs]
        req, got = np.array(req), np.array(got)
        rng = np.random.default_rng(seed)
        per = {vocab[c]: float((got[req == c] == c).mean()) for c in range(len(vocab))}
        res[name] = {"accuracy": float((req == got).mean()),
                     "shuffled_baseline": float((rng.permutation(req) == got).mean()),
                     "chance_uniform": 1 / len(vocab), "per_class": per}
    return res


def _rmse_matrix(a: torch.Tensor, b: torch.Tensor, chunk: int = 512) -> torch.Tensor:
    """Per-pixel RMSE between every image in a and every image in b (values in [0,1])."""
    a2, b2 = a.flatten(1), b.flatten(1)
    d = a2.shape[1]
    rows = [torch.cdist(a2[i:i + chunk], b2) for i in range(0, a2.shape[0], chunk)]
    return torch.cat(rows) / np.sqrt(d)


def _pairwise_stats(x: torch.Tensor) -> dict:
    m = _rmse_matrix(x, x)
    iu = torch.triu_indices(len(x), len(x), 1)
    v = m[iu[0], iu[1]]
    return {"mean": float(v.mean()), "std": float(v.std()), "min": float(v.min()),
            "near_duplicate_pairs(<0.02)": int((v < 0.02).sum())}


def _color_hist(x: torch.Tensor, bins: int = 16) -> np.ndarray:
    h = [torch.histc(x[:, c].flatten(), bins=bins, min=0, max=1) for c in range(3)]
    h = torch.stack(h)
    return (h / h.sum(dim=1, keepdim=True)).numpy()


# --------------------------------------------------------------------------- main evaluation
def evaluate(checkpoint, cfg: Config | None = None, num_samples: int = 512, seed: int = 123,
             output_dir=None, device="auto", data_dir=None) -> dict:
    from dataset import TinyHeroDataset, download_tinyhero
    from generate import interpolate, load_generator
    from utils import make_grid

    device = get_device(device) if isinstance(device, str) else device
    G, ck_cfg, meta = load_generator(checkpoint, device)
    cfg = cfg or ck_cfg
    out = Path(output_dir or cfg.output_dir)
    plots = out / "plots"
    plots.mkdir(parents=True, exist_ok=True)
    root = download_tinyhero(data_dir or cfg.data_dir)
    ds = TinyHeroDataset(root, views=cfg.views, background=cfg.background,
                         with_attributes=bool(cfg.conditional))  # full set, no subset
    real_all = ds.as_float01()
    metrics = {"checkpoint": str(checkpoint), "checkpoint_epoch": meta["epoch"], "device": str(device),
               "num_generated": num_samples, "num_training_images": len(ds), "seed": seed}

    # 1. losses
    hist_csv, iter_csv = out / "training_history.csv", out / "training_iterations.csv"
    if hist_csv.exists():
        plot_losses(hist_csv, iter_csv, plots / "losses.png")
        hist = _read_csv(hist_csv)
        metrics["final_epoch_losses"] = {k: hist[-1][k] for k in ("loss_d", "loss_g", "d_real", "d_fake")}
    elif meta.get("history"):
        tmp = out / "training_history_from_ckpt.csv"
        from train import EPOCH_FIELDS, _rewrite_csv
        _rewrite_csv(tmp, EPOCH_FIELDS, meta["history"])
        plot_losses(tmp, None, plots / "losses.png")

    # 2. progression
    progression_montage(out / "training_progress", plots / "progression.png")

    # 3. real vs generated
    rng0 = np.random.default_rng(seed + 1)
    gen_labels = ds.attrs[rng0.choice(len(ds), num_samples)] if cfg.conditional else None  # data label mix
    gen = generate_batch01(G, num_samples, seed, device, labels=gen_labels)
    rng = np.random.default_rng(seed)
    real_idx = rng.choice(len(ds), size=min(num_samples, len(ds)), replace=False)
    real = real_all[real_idx]
    fig, ax = _fig(12, 6.4, 1, 2)
    _imshow_grid(ax[0, 0], _to_pil(real[:36]), "REAL - TinyHero training images")
    _imshow_grid(ax[0, 1], _to_pil(gen[:36]), f"GENERATED - DCGAN (epoch {meta['epoch']})")
    fig.tight_layout()
    fig.savefig(plots / "real_vs_generated.png", facecolor=SURFACE)

    # 4. interpolation (3 seed pairs, linear vs spherical)
    pairs = [(seed, seed + 1), (seed + 2, seed + 3), (seed + 4, seed + 5)]
    fig, ax = _fig(12, 1.0 * len(pairs) * 2 + 0.6, len(pairs) * 2, 1)
    for i, (a, b) in enumerate(pairs):
        for j, mode in enumerate(("linear", "slerp")):
            frames = interpolate(G, a, b, 10, mode, device)
            axx = ax[i * 2 + j, 0]
            axx.imshow(make_grid(frames, nrow=len(frames)), interpolation="nearest")
            axx.set_xticks([]), axx.set_yticks([])
            for sp in axx.spines.values():
                sp.set_visible(False)
            axx.set_ylabel(f"z{a}->z{b}\n{mode}", color=INK, fontsize=8, rotation=0, ha="right", va="center")
    fig.suptitle("Latent interpolation: 10 decoded points between two seeds", color=INK, fontsize=11, x=0.02, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.97))  # leave room for the suptitle
    fig.savefig(plots / "interpolation_experiment.png", facecolor=SURFACE)
    # smoothness: mean RMSE between consecutive frames vs endpoints distance
    fr = torch.stack([torch.from_numpy(np.array(f)).permute(2, 0, 1).float() / 255
                                          for f in interpolate(G, pairs[0][0], pairs[0][1], 10, "linear", device)])
    step = _rmse_matrix(fr[:-1], fr[1:]).diagonal()
    metrics["interpolation"] = {"pair": list(pairs[0]),
                                "mean_consecutive_frame_rmse": float(step.mean()),
                                "max_consecutive_frame_rmse": float(step.max()),
                                "endpoint_rmse": float(_rmse_matrix(fr[:1], fr[-1:])[0, 0])}

    # 5. diversity & memorisation
    metrics["pairwise_rmse_generated"] = _pairwise_stats(gen)
    metrics["pairwise_rmse_real"] = _pairwise_stats(real)
    metrics["diversity_ratio_generated_over_real"] = (
        metrics["pairwise_rmse_generated"]["mean"] / metrics["pairwise_rmse_real"]["mean"])
    pooled_g, pooled_r = F.avg_pool2d(gen, 4), F.avg_pool2d(real, 4)  # 16x16 coarse structure
    metrics["pairwise_rmse_16x16_generated"] = _pairwise_stats(pooled_g)["mean"]
    metrics["pairwise_rmse_16x16_real"] = _pairwise_stats(pooled_r)["mean"]

    d_gen = _rmse_matrix(gen, real_all)                       # generated -> nearest training image
    nn_gen, nn_idx = d_gen.min(dim=1)
    d_real = _rmse_matrix(real, real_all)                     # real -> nearest OTHER training image
    d_real[torch.arange(len(real_idx)), torch.as_tensor(real_idx)] = float("inf")
    nn_real = d_real.min(dim=1).values
    metrics["nearest_neighbor_rmse_generated_to_train"] = {
        "mean": float(nn_gen.mean()), "median": float(nn_gen.median()), "min": float(nn_gen.min())}
    metrics["nearest_neighbor_rmse_real_to_other_train"] = {
        "mean": float(nn_real.mean()), "median": float(nn_real.median()), "min": float(nn_real.min())}
    metrics["fraction_generated_closer_than_real_median_nn"] = float((nn_gen < nn_real.median()).float().mean())
    metrics["possible_copies(nn_rmse<0.02)"] = int((nn_gen < 0.02).sum())
    hg, hr = _color_hist(gen), _color_hist(real)
    metrics["color_histogram_L1_distance"] = float(np.abs(hg - hr).sum() / 3)

    # nearest-neighbour figure: generated | 3 closest training images
    k, show = 3, 8
    top = d_gen[:show].topk(k, largest=False)
    fig, ax = _fig(2.2 * (k + 1), 2.0 * show + 0.6, show, k + 1)
    for r in range(show):
        tiles = [(gen[r], "generated", None)] + [
            (real_all[top.indices[r, j]], f"NN{j + 1} rmse={top.values[r, j]:.3f}", None) for j in range(k)]
        for c, (img, label, _) in enumerate(tiles):
            a = ax[r, c]
            a.imshow(img.permute(1, 2, 0).numpy(), interpolation="nearest")
            a.set_xticks([]), a.set_yticks([])
            for sp in a.spines.values():
                sp.set_edgecolor(C_GEN if c == 0 else GRID)
                sp.set_linewidth(2 if c == 0 else 1)
            a.set_title(label, color=INK if c == 0 else INK_MUTED, fontsize=8)
    fig.suptitle("Generated sample vs its 3 nearest TinyHero training images (pixel RMSE)",
                 color=INK, fontsize=11, x=0.02, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.97))  # leave room for the suptitle
    fig.savefig(plots / "nearest_neighbors.png", facecolor=SURFACE)

    # distribution figure: two panels, one axis each
    fig, ax = _fig(12, 3.6, 1, 2)
    bins = np.linspace(0, float(max(nn_gen.max(), nn_real.max())) * 1.05, 40)
    a = ax[0, 0]
    a.hist(nn_real.numpy(), bins=bins, color=C_DISC, alpha=0.65, label="real -> nearest other real")
    a.hist(nn_gen.numpy(), bins=bins, color=C_GEN, alpha=0.65, label="generated -> nearest real")
    _style(a, "Nearest-neighbour distance to the training set", "pixel RMSE", "count")
    _legend(a)
    a = ax[0, 1]
    iu = torch.triu_indices(min(len(gen), 300), min(len(gen), 300), 1)
    pg = _rmse_matrix(gen[:300], gen[:300])[iu[0], iu[1]].numpy()
    pr = _rmse_matrix(real[:300], real[:300])[iu[0], iu[1]].numpy()
    bins2 = np.linspace(0, max(pg.max(), pr.max()) * 1.05, 40)
    a.hist(pr, bins=bins2, color=C_DISC, alpha=0.65, label="real pairs")
    a.hist(pg, bins=bins2, color=C_GEN, alpha=0.65, label="generated pairs")
    _style(a, "Pairwise distance within each set (diversity)", "pixel RMSE", "count")
    _legend(a)
    fig.tight_layout()
    fig.savefig(plots / "diversity.png", facecolor=SURFACE)

    # 6. controllability (conditional model only)
    if cfg.conditional:
        from attributes import HAIR, OUTFIT, VIEWS
        ctrl = controllability(G, device, cfg.background)
        metrics["controllability"] = ctrl
        # labeller self-consistency on REAL images (upper bound for what this metric can show)
        from attributes import label_image as _li
        sub_idx = np.random.default_rng(0).choice(len(ds), 400, replace=False)
        real_imgs = _to_pil(real_all[sub_idx])
        for col, name in ((1, "hair"), (2, "outfit")):
            ctrl[name]["labeller_agreement_on_real"] = float(np.mean(
                [_li(im, cfg.background)[name] == int(ds.attrs[i][col]) for im, i in zip(real_imgs, sub_idx)]))
        fig, ax = _fig(12, 3.8, 1, 2)
        for a, name, vocab in ((ax[0, 0], "hair", HAIR), (ax[0, 1], "outfit", OUTFIT)):
            per = ctrl[name]["per_class"]
            xs = np.arange(len(vocab))
            a.bar(xs, [per[v] for v in vocab], color=C_GEN, width=0.6, label="requested colour obtained")
            a.axhline(ctrl[name]["shuffled_baseline"], color=C_DISC, linewidth=2, label="baseline (requests ignored)")
            a.set_xticks(xs, vocab, rotation=30, color=INK_MUTED, fontsize=8)
            a.set_ylim(0, 1)
            _style(a, f"{name.title()} control: {ctrl[name]['accuracy']:.0%} overall", None, "fraction of samples")
            _legend(a)
        fig.tight_layout()
        fig.savefig(plots / "controllability.png", facecolor=SURFACE)
        # visual grid: rows = hair colours, columns = views x2
        from generate import make_labels
        rows = []
        for h in range(len(HAIR)):
            y = make_labels(G, 8, {"hair": h, "outfit": None}, 500 + h, "cpu")
            y[:, 0] = torch.tensor([0, 1, 2, 3, 0, 1, 2, 3])
            rows += _to_pil(generate_batch01(G, 8, 900 + h, device, labels=y))
        make_grid(rows, nrow=8, scale=3).save(plots / "prompt_control_grid.png")

    metrics["interpretation"] = interpret(metrics)
    metrics["notes"] = [
        "Pixel RMSE is a crude proxy: it ignores perceptual similarity and small shifts.",
        "Diversity ratio near 1 means generated samples are as spread out as real ones; much lower suggests mode collapse.",
        "Generated->train NN distances far below real->train NN distances would suggest memorisation.",
        "FID was not computed: it needs an ImageNet Inception network and thousands of samples, and is "
        "unreliable for 64x64 pixel art with ~3.6k training images.",
    ]
    (out / "evaluation_metrics.json").write_text(json.dumps(metrics, indent=2))
    return metrics


def interpret(m: dict) -> list[str]:
    """Plain-language reading of the computed numbers (rules are fixed, numbers are measured)."""
    out = []
    r = m["diversity_ratio_generated_over_real"]
    if r < 0.5:
        out.append(f"Diversity ratio {r:.2f}: generated images are much less varied than real ones - likely mode collapse.")
    elif r < 0.85:
        out.append(f"Diversity ratio {r:.2f}: somewhat less varied than the real data (partial mode dropping is possible).")
    else:
        out.append(f"Diversity ratio {r:.2f}: generated samples are about as varied as real samples in pixel space.")
    g = m["nearest_neighbor_rmse_generated_to_train"]["median"]
    rr = m["nearest_neighbor_rmse_real_to_other_train"]["median"]
    copies = m["possible_copies(nn_rmse<0.02)"]
    for name, c in m.get("controllability", {}).items():
        out.append(f"{name.title()} control: requested colour obtained in {c['accuracy']:.0%} of samples vs "
                   f"{c['shuffled_baseline']:.0%} if requests were ignored (labeller agrees with itself on "
                   f"{c.get('labeller_agreement_on_real', float('nan')):.0%} of real images).")
    if copies > 0:
        out.append(f"{copies} generated samples are within RMSE 0.02 of a training image - inspect them for copying.")
    if g < 0.8 * rr:
        out.append(f"Median NN distance generated->train ({g:.3f}) is clearly below real->train ({rr:.3f}): "
                   "samples sit unusually close to training images (possible memorisation).")
    else:
        out.append(f"Median NN distance generated->train ({g:.3f}) vs real->other-train ({rr:.3f}): "
                   "no sign of copying; samples are new combinations rather than near-duplicates.")
    return out


def main():
    p = argparse.ArgumentParser(description="Evaluate a trained PixelForge generator")
    p.add_argument("--checkpoint", help="default: checkpoints/generator_final.pt (or latest.pt)")
    p.add_argument("--num-samples", type=int, default=512)
    p.add_argument("--seed", type=int, default=123)
    p.add_argument("--output-dir", default=str(PROJECT_ROOT / "outputs"))
    p.add_argument("--data-dir", default=str(PROJECT_ROOT / "data"))
    p.add_argument("--device", default="auto", choices=["auto", "cuda", "mps", "cpu"])
    args = p.parse_args()
    from generate import find_checkpoint
    ckpt = Path(args.checkpoint) if args.checkpoint else find_checkpoint()
    if ckpt is None or not ckpt.exists():
        raise SystemExit("No checkpoint found. Train first or pass --checkpoint PATH.")
    m = evaluate(ckpt, num_samples=args.num_samples, seed=args.seed, output_dir=args.output_dir,
                 device=args.device, data_dir=args.data_dir)
    print(json.dumps(m, indent=2))
    print(f"Plots in {Path(args.output_dir) / 'plots'}; metrics in {Path(args.output_dir) / 'evaluation_metrics.json'}")


if __name__ == "__main__":
    main()

In [ ]:
evaluate = fresh("evaluate")
out = Path(cfg.output_dir)
loss_png = evaluate.plot_losses(out / "training_history.csv", out / "training_iterations.csv", out / "plots" / "losses.png")
display(IPImage(str(loss_png)))
import pandas as pd
hist_df = pd.read_csv(out / "training_history.csv")
display(hist_df.tail(10))

## 16. Training progress visualizations

The same 64 latent vectors are decoded after different numbers of epochs. Look for: blobs → silhouettes →
heads/bodies/legs → coloured clothing, hair and equipment.

In [ ]:
prog_png = evaluate.progression_montage(out / "training_progress", out / "plots" / "progression.png")
if prog_png:
    display(IPImage(str(prog_png)))
final_grid = sorted((out / "training_progress").glob("epoch_*.png"))[-1]
print("Final fixed-noise grid:", final_grid.name)
display(IPImage(str(final_grid)))

## 17. New character generation

Load the inference-only `generator_final.pt` (CPU first via `map_location="cpu"`, then moved to the device),
sample new latent vectors with a fixed seed and decode them. Outputs are converted from $[-1,1]$ to $[0,255]$ and
saved as genuine **RGB** PNGs; previews use nearest-neighbour upscaling to keep pixel edges crisp.
Transparent copies (optional) are made with a documented flood-fill heuristic, **not** learned by the model.

In [ ]:
%%writefile generate.py
"""Generate new characters (and latent interpolations) from a trained Generator.

Examples
    python generate.py --num-images 16 --seed 7
    python generate.py --num-images 9 --scale 8 --transparent
    python generate.py --interpolate --seed-a 1 --seed-b 2 --steps 10 --gif
"""
from __future__ import annotations

import argparse
from pathlib import Path

import torch
from PIL import Image

from config import PROJECT_ROOT, Config, get_device
from models import ConditionalGenerator, Generator, interpolate_latents, sample_latent
from utils import make_grid, remove_background, save_gif, tensor_to_pil, upscale

DEFAULT_CANDIDATES = ("generator_final.pt", "latest.pt")


def find_checkpoint(checkpoint_dir: str | Path = PROJECT_ROOT / "checkpoints") -> Path | None:
    d = Path(checkpoint_dir)
    for name in DEFAULT_CANDIDATES:
        if (d / name).exists():
            return d / name
    eps = sorted(d.glob("pixelforge_epoch_*.pt"))
    return eps[-1] if eps else None


def load_generator(path: str | Path, device: torch.device | str = "cpu"):
    """Load a Generator from either a full training checkpoint or a generator-only file.

    Files are loaded with map_location="cpu" and then moved to `device`, so a
    checkpoint trained on a Colab T4 (CUDA) works on a Mac (MPS) or CPU.
    Returns (generator in eval mode, Config, checkpoint metadata dict).
    """
    try:
        ck = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        ck = torch.load(path, map_location="cpu")
    if not isinstance(ck, dict) or "generator" not in ck:
        raise ValueError(f"{path} does not look like a PixelForge checkpoint (no 'generator' key).")
    cfg = Config.from_dict(ck.get("config", {}))
    G = ConditionalGenerator(cfg.nz, cfg.ngf, n_classes=cfg.n_classes) if cfg.conditional else Generator(cfg.nz, cfg.ngf)
    G.load_state_dict(ck["generator"])
    G.to(device).eval()
    meta = {k: ck.get(k) for k in ("epoch", "format", "params", "history", "ema")}
    meta["path"] = str(path)
    return G, cfg, meta


def is_conditional(G) -> bool:
    return bool(getattr(G, "conditional", False))


def make_labels(G, n: int, labels: dict | None = None, seed: int | None = None,
                device: torch.device | str = "cpu") -> torch.Tensor | None:
    """(n, 3) label tensor [view, hair, outfit]. Unspecified attributes (None) are sampled with `seed`."""
    if not is_conditional(G):
        return None
    labels = labels or {}
    g = torch.Generator().manual_seed(int(seed) + 7919 if seed is not None else 0)
    cols = []
    for i, (name, ncls) in enumerate(zip(("view", "hair", "outfit"), G.n_classes)):
        v = labels.get(name)
        cols.append(torch.full((n,), int(v)) if v is not None else torch.randint(0, ncls, (n,), generator=g))
    return torch.stack(cols, dim=1).long().to(device)


@torch.no_grad()
def generate(G: Generator, n: int, seed: int | None = None, device: torch.device | str = "cpu",
             z: torch.Tensor | None = None, labels: dict | torch.Tensor | None = None) -> list[Image.Image]:
    """Sample n latent vectors (deterministic if seed is set) and return 64x64 RGB PIL images.

    For a conditional generator, `labels` is a dict like {"view": 2, "hair": 6, "outfit": None}
    (None/missing = random) or a ready (n, 3) tensor.
    """
    if z is None:
        z = sample_latent(n, G.nz, seed=seed, device=device)
    if not is_conditional(G):
        return tensor_to_pil(G(z.to(device)))
    y = labels.to(device) if torch.is_tensor(labels) else make_labels(G, n, labels, seed, device)
    return tensor_to_pil(G(z.to(device), y))


@torch.no_grad()
def interpolate(G: Generator, seed_a: int, seed_b: int, steps: int = 10, mode: str = "linear",
                device: torch.device | str = "cpu", labels: dict | None = None) -> list[Image.Image]:
    """Decode the path between the latent vectors produced by seed_a and seed_b.

    Conditional G: attributes are held fixed along the path (from `labels`, else drawn with seed_a).
    """
    z1 = sample_latent(1, G.nz, seed=seed_a, device=device)[0]
    z2 = sample_latent(1, G.nz, seed=seed_b, device=device)[0]
    zs = interpolate_latents(z1, z2, steps, mode)
    if not is_conditional(G):
        return tensor_to_pil(G(zs))
    y = make_labels(G, 1, labels, seed_a, device).repeat(steps, 1)
    return tensor_to_pil(G(zs, y))


def save_images(images: list[Image.Image], out_dir: str | Path, prefix: str = "character",
                scale: int = 1, transparent: bool = False, background=(0, 0, 0)) -> list[Path]:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    paths = []
    for i, im in enumerate(images):
        p = out / f"{prefix}_{i:03d}.png"
        upscale(im, scale).save(p)  # genuine RGB output of the GAN
        paths.append(p)
        if transparent:
            upscale(remove_background(im, background), scale).save(out / f"{prefix}_{i:03d}_transparent.png")
    return paths


def main():
    p = argparse.ArgumentParser(description="Generate PixelForge characters from a trained DCGAN")
    p.add_argument("--checkpoint", help="path to generator_final.pt / latest.pt (default: auto-detect in checkpoints/)")
    p.add_argument("--num-images", type=int, default=16, help="number of characters (4, 9 and 16 make square grids)")
    p.add_argument("--seed", type=int, default=None, help="fix for deterministic output")
    p.add_argument("--output-dir", default=str(PROJECT_ROOT / "outputs" / "generated"))
    p.add_argument("--scale", type=int, default=4, help="nearest-neighbour upscale for the preview grid")
    p.add_argument("--sprite-scale", type=int, default=1, help="upscale for individual sprite PNGs (1 = native 64x64)")
    p.add_argument("--device", default="auto", choices=["auto", "cuda", "mps", "cpu"])
    p.add_argument("--transparent", action="store_true",
                   help="ALSO save background-removed copies (heuristic flood fill, see README)")
    p.add_argument("--interpolate", action="store_true", help="make a latent interpolation strip instead")
    p.add_argument("--seed-a", type=int, default=0)
    p.add_argument("--seed-b", type=int, default=1)
    p.add_argument("--steps", type=int, default=10)
    p.add_argument("--mode", default="linear", choices=["linear", "slerp"])
    p.add_argument("--gif", action="store_true", help="also save an animated GIF of the interpolation")
    p.add_argument("--interp-dir", default=str(PROJECT_ROOT / "outputs" / "interpolation"))
    args = p.parse_args()

    device = get_device(args.device)
    ckpt = Path(args.checkpoint) if args.checkpoint else find_checkpoint()
    if ckpt is None or not ckpt.exists():
        raise SystemExit("No checkpoint found. Train first (python train.py --epochs 50, or the Colab notebook) "
                         "and put generator_final.pt in checkpoints/, or pass --checkpoint PATH.")
    G, cfg, meta = load_generator(ckpt, device)
    print(f"Loaded {ckpt} (epoch {meta['epoch']}) on {device}")

    if args.interpolate:
        out = Path(args.interp_dir)
        frames = interpolate(G, args.seed_a, args.seed_b, args.steps, args.mode, device)
        out.mkdir(parents=True, exist_ok=True)
        tag = f"seed{args.seed_a}_to_seed{args.seed_b}_{args.mode}"
        strip = make_grid(frames, nrow=len(frames), scale=args.scale)
        strip.save(out / f"interpolation_{tag}.png")
        print(f"Saved {out / f'interpolation_{tag}.png'}")
        if args.gif:
            g = save_gif([upscale(f, args.scale) for f in frames], out / f"interpolation_{tag}.gif")
            print(f"Saved {g}")
        return

    images = generate(G, args.num_images, args.seed, device)
    tag = f"seed{args.seed}" if args.seed is not None else "random"
    paths = save_images(images, Path(args.output_dir) / tag, scale=args.sprite_scale,
                        transparent=args.transparent, background=cfg.background)
    grid = make_grid(images, scale=args.scale)
    grid_path = Path(args.output_dir) / f"grid_{args.num_images}_{tag}.png"
    grid.save(grid_path)
    print(f"Saved {len(paths)} sprites to {Path(args.output_dir) / tag} and grid {grid_path}")


if __name__ == "__main__":
    main()

In [ ]:
generate = fresh("generate")
G_inf, cfg_inf, meta = generate.load_generator(Path(cfg.checkpoint_dir) / "generator_final.pt", device)
print("Loaded generator from epoch", meta["epoch"])
GEN_SEED = 2024
chars = generate.generate(G_inf, NUM_GENERATED, seed=GEN_SEED, device=device)
gen_dir = out / "generated"
paths = generate.save_images(chars, gen_dir / f"seed{GEN_SEED}", transparent=True, background=cfg.background)
grid = utils.make_grid(chars, scale=4)
grid.save(gen_dir / f"grid_{NUM_GENERATED}_seed{GEN_SEED}.png")
print(f"Saved {len(paths)} sprites to {gen_dir / f'seed{GEN_SEED}'}")
display(grid)

### 17b. Prompt → character

`parse_prompt` turns the description into labels; unspecified attributes are random. Each row below is one
prompt, decoded with 8 different latent vectors $z$ (same labels, different characters). **Edit `PROMPTS` to try your own.**

In [ ]:
PROMPTS = ["a front-facing hero with green hair and a red tunic",
           "knight in armour seen from behind",
           "blonde adventurer in a blue outfit, facing left",
           "purple-haired mage in a white robe",
           "red-haired rogue in a brown leather vest, right side"]
if getattr(G_inf, "conditional", False):
    prompt_dir = out / "generated" / "prompts"; prompt_dir.mkdir(parents=True, exist_ok=True)
    for k, p in enumerate(PROMPTS):
        labels, notes = attributes.parse_prompt(p)
        imgs = generate.generate(G_inf, 8, seed=100 + k, device=device, labels=labels)
        row = utils.make_grid(imgs, nrow=8, scale=3)
        row.save(prompt_dir / f"prompt_{k}.png")
        print(f"{p!r}  ->  {', '.join(notes)}"); display(row)
else:
    print("This generator is unconditional (CONDITIONAL = False): prompts cannot be used.")

## 18. Latent space interpolation

Pick two latent vectors $z_1, z_2$ and decode points along the line between them:
$z(t) = (1-t)\,z_1 + t\,z_2$. Because $G$ is a continuous function, nearby latent vectors map to related images,
so a well-trained generator shows a gradual morph (body shape, colours, equipment) rather than random jumps.
We also show **slerp** (spherical interpolation), which keeps $\|z\|$ close to the typical norm of a 100-d Gaussian.
Not every transition is semantically meaningful — some intermediate frames can look less coherent.

In [ ]:
interp_dir = out / "interpolation"
interp_dir.mkdir(parents=True, exist_ok=True)
for a, b in [(7, 2024), (1, 99), (42, 314)]:
    frames = generate.interpolate(G_inf, a, b, steps=10, mode="linear", device=device)
    strip = utils.make_grid(frames, nrow=len(frames), scale=3)
    strip.save(interp_dir / f"interpolation_seed{a}_to_seed{b}_linear.png")
    utils.save_gif([utils.upscale(f, 4) for f in frames], interp_dir / f"interpolation_seed{a}_to_seed{b}.gif")
    print(f"seed {a} -> seed {b} (linear)"); display(strip)
frames = generate.interpolate(G_inf, 7, 2024, steps=10, mode="slerp", device=device)
print("seed 7 -> seed 2024 (slerp)"); display(utils.make_grid(frames, nrow=10, scale=3))
display(IPImage(str(interp_dir / "interpolation_seed7_to_seed2024.gif")))

## 19. Evaluation

Five experiments, all computed from this run (nothing is hard-coded):

1. **Training loss** — Section 15.
2. **Visual quality progression** — Section 16.
3. **Real vs generated** — side-by-side, clearly labelled grids.
4. **Latent interpolation** — strips for 3 seed pairs, linear vs slerp, plus frame-to-frame smoothness.
5. **Diversity & memorisation** — 512 generated vs 512 real images:
   * mean pairwise pixel RMSE within each set (diversity ratio ≈ 1 is good, ≪ 1 suggests mode collapse);
   * nearest-neighbour distance from each generated image to the **whole training set**, compared with the
     distance from real images to their nearest *other* training image (much smaller → memorisation);
   * count of near-copies (RMSE < 0.02) and colour-histogram distance.

**FID is intentionally not computed**: it requires an ImageNet-pretrained Inception network and many thousands
of samples, and is unreliable for 64×64 pixel art with only 3.6k training images.

In [ ]:
metrics = evaluate.evaluate(Path(cfg.checkpoint_dir) / "generator_final.pt", cfg, num_samples=512,
                            output_dir=out, device=device)
for name in ["controllability", "prompt_control_grid", "real_vs_generated", "interpolation_experiment",
             "nearest_neighbors", "diversity"]:
    if (out / "plots" / f"{name}.png").exists():
        print(name); display(IPImage(str(out / "plots" / f"{name}.png")))

## 20. Results

The table below is filled in from `evaluation_metrics.json` produced by the cell above — for **this** run.
If the notebook ran in smoke-test mode, these numbers describe a barely-trained model and must not be reported
as final results.

In [ ]:
print(f"Mode: {MODE} | epochs trained: {metrics['checkpoint_epoch']} | device: {metrics['device']}")
rows = {
    "Final loss_D / loss_G": f"{metrics.get('final_epoch_losses', {}).get('loss_d', float('nan')):.3f} / "
                             f"{metrics.get('final_epoch_losses', {}).get('loss_g', float('nan')):.3f}",
    "Mean pairwise RMSE generated": f"{metrics['pairwise_rmse_generated']['mean']:.4f}",
    "Mean pairwise RMSE real": f"{metrics['pairwise_rmse_real']['mean']:.4f}",
    "Diversity ratio (gen/real)": f"{metrics['diversity_ratio_generated_over_real']:.3f}",
    "Median NN RMSE generated->train": f"{metrics['nearest_neighbor_rmse_generated_to_train']['median']:.4f}",
    "Median NN RMSE real->other train": f"{metrics['nearest_neighbor_rmse_real_to_other_train']['median']:.4f}",
    "Near-copies (NN RMSE < 0.02)": metrics["possible_copies(nn_rmse<0.02)"],
    "Colour histogram L1 distance": f"{metrics['color_histogram_L1_distance']:.4f}",
    "Interpolation: mean frame-to-frame RMSE": f"{metrics['interpolation']['mean_consecutive_frame_rmse']:.4f}",
}
for name, c in metrics.get("controllability", {}).items():
    rows[f"{name.title()} control: requested colour obtained"] = f"{c['accuracy']:.1%}"
    rows[f"{name.title()} control: if requests ignored (baseline)"] = f"{c['shuffled_baseline']:.1%}"
display(pd.DataFrame(rows.items(), columns=["metric", "value"]))
print("\nInterpretation (rules applied to the measured numbers):")
for line in metrics["interpretation"]:
    print(" -", line)

## 21. Limitations

* **Small dataset** (3,648 images, 912 unique characters) → limited variety; four views of each character are
  strongly correlated.
* **Unconditional model** — you cannot request a direction, class or item. A conditional GAN would add that.
* **64×64 resolution** and DCGAN capacity: details such as faces and weapons are often blurry or incoherent;
  transposed convolutions can create checkerboard artifacts.
* **GAN instability** — results depend on seeds and hyperparameters; mode collapse is possible.
* **Evaluation metrics are simple** pixel-space proxies; they do not measure perceptual quality.
* **Transparency** is not learned: the optional background removal is a heuristic flood fill.

## 22. Conclusion

We trained a conditional DCGAN from scratch on TinyHero and used it to create new pixel-art RPG characters from
short text descriptions.
The notebook verifies the dataset, implements the standard DCGAN architecture and training procedure with
logging, checkpoints and resume support, and evaluates the model with loss curves, fixed-noise progression,
real-vs-generated grids, latent interpolation, and diversity / nearest-neighbour memorisation checks.
The exported `generator_final.pt` runs on CUDA, Apple MPS or CPU and powers the PixelForge Streamlit studio.

## 23. Viva questions and answers (short version — see `VIVA_GUIDE.md` for 25+ detailed answers)

**Q: How does a text prompt become an image without a text model?** The prompt is parsed by keyword rules into
the three labels the GAN was trained on (view, hair colour, outfit colour). The conditional generator $G(z, y)$
draws a character with those labels; $z$ supplies the variation. Words outside that vocabulary are ignored.

**Q: Where do the labels come from?** View comes from the dataset folders. Hair and outfit colour are measured
automatically from the pixels (majority HSV colour of fixed head and torso regions), so they're approximate.

**Q: How do you know the prompt actually controls the output?** Controllability experiment: request each
colour, label the generated sprites with the same automatic labeller, and compare the hit rate with a baseline
where the requests are shuffled (a model that ignores them).

**Q: What does the generator learn?** A mapping from a simple distribution $\mathcal{N}(0,I)$ to the distribution
of sprites, so that samples it produces are indistinguishable (to D) from real ones.

**Q: Why detach the fake images in the D step?** So the discriminator loss does not compute gradients for the
generator — each network is updated only by its own objective.

**Q: Why no sigmoid in D?** `BCEWithLogitsLoss` applies the sigmoid internally in a numerically stable way.

**Q: Why Tanh + [-1, 1] normalisation?** Tanh outputs lie in [-1, 1]; scaling real data to the same range lets
D compare like with like and keeps inputs zero-centred.

**Q: What is mode collapse?** G produces only a few kinds of output that fool D, ignoring the rest of the data
distribution. We check for it with the diversity ratio and by inspecting samples.

**Q: How do you know the model is not copying training images?** Nearest-neighbour search over all 3,648
training images: generated samples are compared with their closest real image, and those distances are compared
with real-to-real nearest-neighbour distances.

**Q: Why use latent interpolation?** Smooth transitions indicate a continuous, structured latent space rather than
a lookup table of memorised sprites.

## 📦 Export results

Packages the checkpoints, history, plots, progression grids, generated sprites and interpolations into
`pixelforge_results.zip`, mirroring the project layout. On your Mac, unzip it **inside the project folder**:

```bash
cd pixelforge
unzip ~/Downloads/pixelforge_results.zip -d .
streamlit run app.py
```

In [ ]:
import zipfile
zip_path = PROJECT_DIR / "pixelforge_results.zip"
ck = Path(cfg.checkpoint_dir)
files = [ck / "generator_final.pt"] + ([ck / "latest.pt"] if INCLUDE_FULL_CHECKPOINT_IN_ZIP else [])
files += [p for p in out.rglob("*") if p.is_file()]
files.append(Path("outputs/dataset_stats.json"))
files += [p for p in Path("outputs/plots").glob("dataset_*.png")]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in dict.fromkeys(files):
        if f.exists():
            zf.write(f, Path(f).resolve().relative_to(PROJECT_DIR))
print(f"{zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB), {len(zipfile.ZipFile(zip_path).namelist())} files")

if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    dst = Path("/content/drive/MyDrive/PixelForge")
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copy(zip_path, dst / zip_path.name)
    print("Copied to", dst / zip_path.name)

if IN_COLAB:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))
else:
    print("Not running in Colab - the ZIP is at", zip_path)